In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ahmedmohsen2005/final-dataset")

print("Path to dataset files:", path)

Mounting files to /kaggle/input/datasets/ahmedmohsen2005/final-dataset...
Path to dataset files: /kaggle/input/datasets/ahmedmohsen2005/final-dataset


In [18]:
#!/usr/bin/env python
# coding: utf-8

# # Notebook 1: DaViT-Tiny + Gated Fusion
# ## Multimodal Explainable AI for Skin Cancer Classification
# ### Model: Dual Attention Vision Transformer (DaViT-Tiny) + Gated Metadata Fusion
#
# **Authors:** Research Pipeline | **Dataset:** ISIC 2017/DICM-17K Style
#
# ---
# ## Scientific Overview
#
# This notebook implements a **publication-quality multimodal AI pipeline** for binary skin
# lesion classification (Melanoma vs. Non-Melanoma). The core architecture couples:
#
# - **DaViT-Tiny**: A Dual Attention Vision Transformer that alternates channel attention
#   and spatial attention, capturing both fine-grained texture (critical for border
#   irregularity, pigmentation) and global lesion structure.
# - **Gated Fusion**: A learned, sigmoid-gated mechanism that dynamically weights the
#   contribution of clinical metadata vs. image features — crucial because metadata
#   reliability varies per patient (missing values, imprecise age, etc.).
#
# **Why this matters clinically**: Melanoma diagnosis depends on both dermoscopic
# appearance (ABCDE rule) AND patient risk factors (age, site, history). A static
# concatenation would treat metadata as equally reliable as the image — Gated Fusion
# respects the heterogeneous nature of clinical data.

# ============================================================
# SECTION 0: Environment Setup & Imports
# ============================================================

import os
import sys
import zipfile
import random
import logging
import warnings
import hashlib
import json
import math
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple, Union
from copy import deepcopy
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.gridspec import GridSpec
import seaborn as sns
from PIL import Image, ImageFilter
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR

import torchvision
import torchvision.transforms as transforms
from torchvision.transforms import functional as TF

from sklearn.preprocessing import LabelEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    roc_curve, precision_recall_curve, brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import StratifiedGroupKFold

import shap
from scipy import stats
from scipy.ndimage import zoom

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ── Reproducibility ──────────────────────────────────────────
SEED = 42
def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Device: {DEVICE} | PyTorch: {torch.__version__}")

# ── Global Configuration ─────────────────────────────────────
CFG = {
    # Paths
    'data_root': '/kaggle/input/datasets/ahmedmohsen2005/final-dataset',
    'output_dir': 'outputs/notebook1_davit',

    # Image
    'img_size': 224,
    'num_channels': 3,

    # Model
    'model_name': 'davit_tiny',
    'embed_dim': 96,
    'num_heads': [3, 6, 12, 24],
    'depths': [1, 1, 3, 1],
    'img_feature_dim': 768,
    'metadata_embed_dim': 128,
    'fusion_dim': 256,
    'num_classes': 2,

    # Training
    'epochs': 2,
    'batch_size': 8,
    'grad_accum_steps': 8,
    'lr': 3e-4,
    'min_lr': 1e-6,
    'weight_decay': 0.05,
    'warmup_epochs': 5,
    'clip_grad': 1.0,
    'label_smoothing': 0.1,
    'dropout': 0.3,
    'stoch_depth_rate': 0.1,
    'ema_decay': 0.9998,
    'patience': 12,

    # SAM
    'use_sam': False,
    'sam_rho': 0.05,

    # Augmentation
    'mixup_alpha': 0.4,
    'cutmix_alpha': 1.0,
    'mixup_prob': 0.5,

    # Focal Loss
    'focal_gamma': 2.0,
    'focal_alpha': 0.75,

    # MC Dropout
    'mc_dropout_samples': 30,

    # TTA
    'tta_steps': 8,

    # XAI
    'xai_samples': 20,
    'cam_layer': 'image_encoder.norm',
}

os.makedirs(CFG['output_dir'], exist_ok=True)
os.makedirs(f"{CFG['output_dir']}/figures", exist_ok=True)
os.makedirs(f"{CFG['output_dir']}/checkpoints", exist_ok=True)


# ============================================================
# SECTION 1: Data Loading — ZIP-Safe Pipeline
# ============================================================
#
# CLINICAL NOTE ON DATA LEAKAGE:
# Patient-level leakage is the most dangerous form of data contamination in medical AI.
# If the same patient's lesion images appear in both training and test sets (e.g., multiple
# lesions from one patient), the model learns patient-specific features (skin tone, hair
# pattern) rather than lesion-level features. This inflates AUC dramatically but fails
# catastrophically on new patients — exactly the deployment scenario that matters most.
# We prevent this by: (1) tracking patient_id across splits, (2) asserting zero overlap,
# (3) using StratifiedGroupKFold for cross-validation if retraining is needed.

class ZipDataLoader:
    """
    Safely extracts and validates ZIP-structured ISIC datasets.
    Handles corrupted files, missing images, schema inconsistencies.
    """

    REQUIRED_COLS = ['image', 'class']
    SUPPORTED_EXTS = {'.jpg', '.jpeg', '.png', '.bmp'}

    def __init__(self, data_root: str):
        self.data_root = Path(data_root)
        self.splits = {}
        self.extraction_log = defaultdict(list)

    def _safe_extract(self, zip_path: Path, extract_to: Path) -> bool:
        """Extract ZIP with safety checks (no path traversal, no overwrites)."""
        if not zip_path.exists():
            logger.warning(f"ZIP not found: {zip_path}")
            return False

        extract_to.mkdir(parents=True, exist_ok=True)
        try:
            with zipfile.ZipFile(zip_path, 'r') as zf:
                # Security: prevent path traversal
                for member in zf.namelist():
                    target = extract_to / member
                    if not str(target.resolve()).startswith(str(extract_to.resolve())):
                        logger.error(f"Path traversal attempt: {member}")
                        continue
                    zf.extract(member, extract_to)
            logger.info(f"Extracted: {zip_path.name} → {extract_to}")
            return True
        except zipfile.BadZipFile:
            logger.error(f"Corrupted ZIP: {zip_path}")
            return False

    def _load_csv_robust(self, csv_path: Path) -> Optional[pd.DataFrame]:
        """
        Load CSV with schema validation, column mapping, and ID normalization.
        """
        if not csv_path.exists():
            logger.error(f"CSV not found: {csv_path}")
            return None
    
        df = pd.read_csv(csv_path, low_memory=False)
    
        # 1. Normalize column names (lowercase and underscores)
        df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]
    
        # 2. Map required columns to standard names
        # Based on your data, 'isic_id' maps to 'image' and 'class' is the target
        alt_maps = {
            'image': ['image', 'isic_id', 'id', 'filename'],
            'class': ['class', 'target', 'label', 'diagnosis']
        }
    
        for standard_col, alternatives in alt_maps.items():
            if standard_col not in df.columns:
                for alt in alternatives:
                    if alt in df.columns:
                        df.rename(columns={alt: standard_col}, inplace=True)
                        logger.info(f"Mapped '{alt}' → '{standard_col}'")
                        break
    
        # 3. CRITICAL: Normalize image IDs to stems (remove .jpg, .png, etc.)
        # This fixes the 'Matched 0/11941 images' error by ensuring CSV IDs match file stems
        if 'image' in df.columns:
            df['image'] = df['image'].apply(lambda x: Path(str(x)).stem.strip())
    
        # 4. Ensure the 'class' target is a binary integer
        if 'class' in df.columns:
            if df['class'].dtype == object:
                # Map common string labels to binary 0/1
                mapping = {
                    'melanoma': 1, 'mel': 1, 'yes': 1, '1': 1, '1.0': 1,
                    'non-melanoma': 0, 'nv': 0, 'bkl': 0, 'no': 0, '0': 0, '0.0': 0
                }
                df['class'] = df['class'].str.lower().map(mapping).fillna(0).astype(int)
            else:
                df['class'] = df['class'].fillna(0).astype(int)
    
        # Log results for transparency
        melanoma_count = df['class'].sum() if 'class' in df.columns else 'N/A'
        logger.info(f"CSV loaded: {csv_path.name} | Rows: {len(df)} | Melanoma samples: {melanoma_count}")
        
        return df

    def _scan_images(self, img_dir: Path) -> Dict[str, Path]:
        """Build image → path mapping, skip corrupted files."""
        img_map = {}
        corrupted = []

        for p in img_dir.rglob('*'):
            if p.suffix.lower() not in self.SUPPORTED_EXTS:
                continue

            # Quick integrity check
            try:
                with Image.open(p) as img:
                    img.verify()
                stem = p.stem.strip()
                img_map[stem] = p
            except Exception as e:
                corrupted.append(str(p))
                self.extraction_log['corrupted'].append(str(p))

        if corrupted:
            logger.warning(f"Corrupted images skipped: {len(corrupted)}")
        logger.info(f"Valid images found: {len(img_map)}")
        return img_map

    def _compute_image_hash(self, img_path: Path) -> str:
        """MD5 hash for duplicate detection."""
        h = hashlib.md5()
        with open(img_path, 'rb') as f:
            for chunk in iter(lambda: f.read(8192), b''):
                h.update(chunk)
        return h.hexdigest()

    def load_split(self, split_name: str) -> Optional[Dict]:
        """
        Load a single split (train/val/test) from ZIP.
        Returns dict with 'df', 'img_map', 'img_dir'.
        """
        zip_path = self.data_root / f"{split_name}.zip"
        extract_to = self.data_root / 'extracted' / split_name

        # Check if already extracted
        if not extract_to.exists():
            success = self._safe_extract(zip_path, extract_to)
            if not success:
                # Try direct directory (no ZIP)
                direct_path = self.data_root / split_name
                if direct_path.exists():
                    extract_to = direct_path
                    logger.info(f"Using pre-extracted directory: {direct_path}")
                else:
                    return None

        # Find CSV and images directory
        # Flexible search: handle nested directories
        csv_candidates = list(extract_to.rglob('*.csv'))
        img_candidates = [p for p in extract_to.rglob('images') if p.is_dir()]

        if not csv_candidates:
            logger.error(f"No CSV found in {extract_to}")
            return None

        # Pick the most relevant CSV
        csv_path = sorted(csv_candidates, key=lambda p: len(p.parts))[0]
        img_dir = img_candidates[0] if img_candidates else extract_to / split_name / 'images'

        df = self._load_csv_robust(csv_path)
        if df is None:
            return None

        img_map = self._scan_images(img_dir)

        # Match images to metadata rows
        matched = df['image'].isin(img_map).sum()
        logger.info(f"[{split_name}] Matched {matched}/{len(df)} images")

        if matched < len(df) * 0.5:
            logger.warning(f"[{split_name}] Less than 50% image-metadata matches! Check paths.")

        return {'df': df, 'img_map': img_map, 'img_dir': img_dir, 'split': split_name}

    def load_all(self) -> Dict:
        """Load train/val/test splits."""
        for split in ['train', 'val', 'test']:
            result = self.load_split(split)
            if result is not None:
                self.splits[split] = result
            else:
                logger.warning(f"Split '{split}' could not be loaded — creating synthetic demo data")
                self.splits[split] = self._create_synthetic_split(split)

        self._validate_no_leakage()
        return self.splits

    def _create_synthetic_split(self, split: str) -> Dict:
        """
        Create synthetic ISIC-like data for pipeline demonstration.
        Used when real data is not available.
        """
        n = {'train': 800, 'val': 200, 'test': 200}[split]
        rng = np.random.default_rng(SEED + hash(split) % 1000)

        df = pd.DataFrame({
            'image': [f'ISIC_{split}_{i:06d}' for i in range(n)],
            'target': rng.integers(0, 2, n),
            'age_approx': rng.choice([20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, np.nan], n),
            'sex': rng.choice(['male', 'female', np.nan], n),
            'anatom_site_general': rng.choice(
                ['head/neck', 'upper extremity', 'lower extremity', 'torso', 'palms/soles', 'oral/genital', np.nan], n),
            'diagnosis_confirm_type': rng.choice(['histopathology', 'single image expert consensus', np.nan], n),
            'patient_id': [f'P{rng.integers(0, n//3):04d}' for _ in range(n)],
        })

        # Synthetic image paths (will be replaced by random tensors in Dataset)
        img_map = {row['image']: None for _, row in df.iterrows()}
        return {'df': df, 'img_map': img_map, 'img_dir': None, 'split': split, 'synthetic': True}

    def _validate_no_leakage(self):
        """
        CRITICAL: Assert no patient-level overlap between train/val/test.
        Also check image-level and lesion-level duplicates.
        """
        logger.info("=" * 60)
        logger.info("LEAKAGE PREVENTION VALIDATION")
        logger.info("=" * 60)

        patient_sets = {}
        image_sets = {}

        for split, data in self.splits.items():
            df = data['df']

            # Patient IDs
            if 'patient_id' in df.columns:
                patient_sets[split] = set(df['patient_id'].dropna().unique())

            # Image IDs
            image_sets[split] = set(df['image'].unique())

        # Check all pairwise overlaps
        splits_list = list(patient_sets.keys())
        for i in range(len(splits_list)):
            for j in range(i+1, len(splits_list)):
                s1, s2 = splits_list[i], splits_list[j]
                if patient_sets.get(s1) and patient_sets.get(s2):
                    overlap = patient_sets[s1] & patient_sets[s2]
                    if overlap:
                        logger.error(
                            f"LEAKAGE DETECTED: {len(overlap)} patients in both {s1} and {s2}! "
                            f"This would inflate evaluation metrics. Remove overlapping patients."
                        )
                    else:
                        logger.info(f"✓ No patient overlap: {s1} ↔ {s2}")

                img_overlap = image_sets.get(s1, set()) & image_sets.get(s2, set())
                if img_overlap:
                    logger.error(f"LEAKAGE: {len(img_overlap)} duplicate images: {s1} ↔ {s2}")
                else:
                    logger.info(f"✓ No image overlap: {s1} ↔ {s2}")

        logger.info("Leakage validation complete.")


# ============================================================
# SECTION 2: Medical Image Preprocessing
# ============================================================
#
# Each step is clinically motivated:
# 1. HAIR REMOVAL: Dermoscopic hair obscures lesion borders, causing CNNs to learn hair
#    texture instead of lesion features — a dangerous shortcut.
# 2. COLOR NORMALIZATION: Different dermoscopes, lighting, and skin tones cause color
#    shift that inflates within-device accuracy but fails across devices/clinics.
# 3. CLAHE: Enhances local contrast of pigmentation patterns (asymmetry/color variation
#    in melanoma) without overexposing bright regions.
# 4. BORDER REMOVAL: Vignetting and black corners introduce distribution artifacts that
#    models exploit as shortcuts (border presence → class label correlation).

class MedicalImagePreprocessor:
    """Clinical-grade dermoscopic image preprocessing pipeline."""

    @staticmethod
    def remove_hair_dullrazor(img: np.ndarray, kernel_size: int = 17) -> np.ndarray:
        """
        DullRazor-inspired hair removal.
        Step 1: Detect dark, thin structures (hair) using black-hat morphological filter.
        Step 2: Threshold to binary mask.
        Step 3: Inpaint masked regions with Telea algorithm.

        Clinical rationale: Hair creates edge artifacts that fool gradient-based CAMs
        into highlighting hair rather than lesion borders.
        """
        if img.ndim == 3:
            gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        else:
            gray = img.copy()

        # Black-hat: highlights dark structures smaller than kernel
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
        blackhat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, kernel)

        # Threshold to find hair
        _, hair_mask = cv2.threshold(blackhat, 10, 255, cv2.THRESH_BINARY)

        # Dilate mask slightly to cover hair edges
        hair_mask = cv2.dilate(hair_mask, np.ones((3,3), np.uint8), iterations=1)

        # Inpaint
        result = cv2.inpaint(img if img.ndim == 3 else cv2.cvtColor(img, cv2.COLOR_GRAY2RGB),
                             hair_mask, inpaintRadius=3, flags=cv2.INPAINT_TELEA)
        return result

    @staticmethod
    def shades_of_gray_normalization(img: np.ndarray, power: float = 6.0) -> np.ndarray:
        """
        Shades of Gray color constancy algorithm.
        Estimates illuminant and normalizes to remove color cast.

        Clinical rationale: Different dermoscope brands and skin tones introduce
        systematic color biases. A melanoma photographed with one dermoscope may
        appear reddish vs. brownish on another, confusing color-sensitive models.
        """
        img_float = img.astype(np.float32) + 1e-6
        norm = np.power(np.mean(np.power(img_float, power), axis=(0, 1)), 1.0 / power)
        scale = np.power(np.prod(norm), 1.0 / 3.0) / norm
        result = np.clip(img_float * scale[np.newaxis, np.newaxis, :], 0, 255).astype(np.uint8)
        return result

    @staticmethod
    def apply_clahe(img: np.ndarray, clip_limit: float = 2.0,
                    tile_size: Tuple[int, int] = (8, 8)) -> np.ndarray:
        """
        CLAHE (Contrast Limited Adaptive Histogram Equalization) in LAB color space.
        Enhances local contrast while preventing over-amplification of noise.

        Clinical rationale: Subtle pigmentation patterns (e.g., regression structures,
        atypical network) in melanoma have low local contrast that standard preprocessing
        misses. CLAHE boosts diagnostically relevant local texture.
        """
        lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
        clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_size)
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    @staticmethod
    def remove_black_border(img: np.ndarray, threshold: int = 15) -> np.ndarray:
        """
        Remove vignetting/dark border artifacts common in dermoscopy.
        Uses contour-based cropping to preserve lesion area.

        Clinical rationale: ~30% of ISIC images have black vignette borders that
        models learn to associate with device type → shortcut learning risk.
        """
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
        _, binary = cv2.threshold(gray, threshold, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            return img
        x, y, w, h = cv2.boundingRect(max(contours, key=cv2.contourArea))
        # Add small padding
        pad = 5
        x, y = max(0, x-pad), max(0, y-pad)
        w = min(img.shape[1]-x, w+2*pad)
        h = min(img.shape[0]-y, h+2*pad)
        return img[y:y+h, x:x+w]

    @staticmethod
    def lesion_centered_crop(img: np.ndarray, output_size: int = 224) -> np.ndarray:
        """
        Attention-guided lesion-centered crop using saliency-proxy (Otsu segmentation).
        Crops a square centered on the most prominent lesion region.

        Clinical rationale: Dermoscopic images often contain skin background 2-3x larger
        than the lesion. Standard resizing shrinks the diagnostically relevant lesion area,
        losing fine-grained features needed for ABCDE criterion evaluation.
        """
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY) if img.ndim == 3 else img
        blurred = cv2.GaussianBlur(gray, (15, 15), 0)
        _, mask = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)

        # Find lesion bounding box
        coords = cv2.findNonZero(mask)
        if coords is None:
            return cv2.resize(img, (output_size, output_size))

        x, y, w, h = cv2.boundingRect(coords)
        cx, cy = x + w//2, y + h//2

        # Square crop centered on lesion
        half = max(w, h) // 2 + 20
        x1 = max(0, cx - half)
        y1 = max(0, cy - half)
        x2 = min(img.shape[1], cx + half)
        y2 = min(img.shape[0], cy + half)

        cropped = img[y1:y2, x1:x2]
        return cv2.resize(cropped, (output_size, output_size))

    def full_pipeline(self, img: np.ndarray, do_hair_removal: bool = True,
                      do_color_norm: bool = True, do_clahe: bool = True,
                      do_border_removal: bool = True,
                      do_lesion_crop: bool = True,
                      output_size: int = 224) -> np.ndarray:
        """Complete preprocessing pipeline."""
        if img is None or img.size == 0:
            return np.zeros((output_size, output_size, 3), dtype=np.uint8)

        # Ensure RGB
        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4:
            img = img[:, :, :3]

        try:
            if do_border_removal:
                img = self.remove_black_border(img)
            if do_hair_removal:
                img = self.remove_hair_dullrazor(img)
            if do_color_norm:
                img = self.shades_of_gray_normalization(img)
            if do_clahe:
                img = self.apply_clahe(img)
            if do_lesion_crop:
                img = self.lesion_centered_crop(img, output_size)
            else:
                img = cv2.resize(img, (output_size, output_size))
        except Exception as e:
            logger.warning(f"Preprocessing step failed: {e}, using raw resize")
            img = cv2.resize(img, (output_size, output_size))

        return img


preprocessor = MedicalImagePreprocessor()


# ============================================================
# SECTION 3: Metadata Preprocessing
# ============================================================

class MetadataPreprocessor:
    """
    Clinical metadata preprocessing with learned embeddings support.
    Handles missing values, encoding, normalization, and feature engineering.
    """

    NUMERICAL_FEATURES = ['age_scaled', 'year']
    CATEGORICAL_FEATURES = ['sex', 'anatom_site_general', 'diagnosis_confirm_type']
    ALL_FEATURES = NUMERICAL_FEATURES + CATEGORICAL_FEATURES

    def __init__(self):
        self.num_imputer = SimpleImputer(strategy='median')
        self.cat_imputer = SimpleImputer(strategy='most_frequent')
        self.scaler = RobustScaler()  # Robust to outliers in clinical data
        self.label_encoders = {}
        self.fitted = False
        self.feature_dims = {}
        self.output_dim = None

    def fit(self, df: pd.DataFrame):
        """Fit on training data only — NEVER fit on val/test."""
        df_work = self._select_available_features(df)

        num_cols = [c for c in self.NUMERICAL_FEATURES if c in df_work.columns]
        cat_cols = [c for c in self.CATEGORICAL_FEATURES if c in df_work.columns]

        if num_cols:
            self.num_imputer.fit(df_work[num_cols])
            num_transformed = self.num_imputer.transform(df_work[num_cols])
            self.scaler.fit(num_transformed)

        for col in cat_cols:
            le = LabelEncoder()
            # Add 'missing' category
            values = df_work[col].fillna('missing').astype(str)
            le.fit(values)
            self.label_encoders[col] = le
            self.feature_dims[col] = len(le.classes_)

        self.available_num = num_cols
        self.available_cat = cat_cols
        self.output_dim = len(num_cols) + len(cat_cols) * 2  # 2 per cat (embed dim / 2 for memory)
        # Actual output: num_cols + one-hot sum
        self._compute_output_dim(df_work)
        self.fitted = True
        logger.info(f"Metadata preprocessor fitted. Output dim: {self.output_dim}")
        logger.info(f"Numerical: {num_cols} | Categorical: {cat_cols}")

    def _compute_output_dim(self, df: pd.DataFrame):
        """Compute final feature vector dimension."""
        dim = len(self.available_num)
        for col in self.available_cat:
            dim += self.feature_dims.get(col, 1)
        self.output_dim = dim

    def _select_available_features(self, df: pd.DataFrame) -> pd.DataFrame:
        available = [c for c in self.ALL_FEATURES if c in df.columns]
        if not available:
            # Create dummy column
            df = df.copy()
            df['age_approx'] = 50.0
            available = ['age_approx']
        return df[available]

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        """Transform a DataFrame to feature matrix."""
        assert self.fitted, "Must call fit() before transform()"
        df_work = self._select_available_features(df)
        n = len(df)
        parts = []

        # Numerical
        if self.available_num:
            num_data = df_work[self.available_num].values.astype(float)
            num_imputed = self.num_imputer.transform(num_data) if num_data.size > 0 else num_data
            num_scaled = self.scaler.transform(num_imputed)
            parts.append(num_scaled)

        # Categorical → one-hot (small cardinality, better for gated fusion)
        for col in self.available_cat:
            if col not in df_work.columns:
                parts.append(np.zeros((n, self.feature_dims.get(col, 1))))
                continue
            values = df_work[col].fillna('missing').astype(str)
            encoded = self.label_encoders[col].transform(
                values.apply(lambda x: x if x in self.label_encoders[col].classes_ else 'missing')
            )
            n_classes = self.feature_dims[col]
            onehot = np.zeros((n, n_classes))
            onehot[np.arange(n), encoded] = 1.0
            parts.append(onehot)

        if parts:
            return np.hstack(parts).astype(np.float32)
        else:
            return np.ones((n, 1), dtype=np.float32) * 0.5  # fallback


meta_preprocessor = MetadataPreprocessor()


# ============================================================
# SECTION 4: Medical Augmentations
# ============================================================

class MedicalAugmentations:
    """
    Clinically safe augmentation pipeline.
    All augmentations must preserve diagnostic validity:
    - Flips are valid (lesions appear in any orientation)
    - Mild color jitter is valid (device/lighting variation)
    - AVOID: extreme color shifts that create non-realistic pigmentation
    - AVOID: heavy blur that removes border irregularity features
    """

    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

    @classmethod
    def get_train_transform(cls, img_size: int = 224) -> transforms.Compose:
        return transforms.Compose([
            transforms.Resize((img_size + 32, img_size + 32)),
            transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0), ratio=(0.9, 1.1)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=30),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1, hue=0.05),
            transforms.RandomGrayscale(p=0.02),
            transforms.RandomApply([transforms.GaussianBlur(kernel_size=3)], p=0.2),
            # MOVE ToTensor() ABOVE RandomErasing
            transforms.ToTensor(), 
            transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
            transforms.Normalize(mean=cls.IMAGENET_MEAN, std=cls.IMAGENET_STD),
        ])

    @classmethod
    def get_val_transform(cls, img_size: int = 224) -> transforms.Compose:
        return transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=cls.IMAGENET_MEAN, std=cls.IMAGENET_STD),
        ])

    @classmethod
    def get_tta_transforms(cls, img_size: int = 224, n: int = 8) -> List[transforms.Compose]:
        """Test-Time Augmentation transforms."""
        base = [
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(mean=cls.IMAGENET_MEAN, std=cls.IMAGENET_STD),
        ]
        tta_list = [transforms.Compose(base)]
        augmentations = [
            [transforms.RandomHorizontalFlip(p=1.0)],
            [transforms.RandomVerticalFlip(p=1.0)],
            [transforms.RandomRotation(degrees=(90, 90))],
            [transforms.RandomRotation(degrees=(180, 180))],
            [transforms.RandomRotation(degrees=(270, 270))],
            [transforms.ColorJitter(brightness=0.1)],
            [transforms.RandomHorizontalFlip(p=1.0), transforms.RandomVerticalFlip(p=1.0)],
        ]
        for aug in augmentations[:n-1]:
            tta_list.append(transforms.Compose(
                [transforms.Resize((img_size, img_size))] + aug +
                [transforms.ToTensor(),
                 transforms.Normalize(mean=cls.IMAGENET_MEAN, std=cls.IMAGENET_STD)]
            ))
        return tta_list


# ============================================================
# SECTION 5: PyTorch Dataset
# ============================================================

class ISICMultimodalDataset(Dataset):
    """
    Multimodal ISIC Dataset returning {image, metadata, label}.
    Supports:
    - Medical image preprocessing pipeline
    - Metadata preprocessing with missing value handling
    - Dynamic augmentation switching
    - Synthetic data fallback
    - Safe image loading with corruption handling
    """

    def __init__(self, split_data: Dict, meta_processor: MetadataPreprocessor,
                 transform=None, img_size: int = 224,
                 apply_medical_preprocessing: bool = True,
                 is_synthetic: bool = False):

        self.df = split_data['df'].copy().reset_index(drop=True)
        self.img_map = split_data['img_map']
        self.transform = transform
        self.img_size = img_size
        self.apply_medical_preprocessing = apply_medical_preprocessing
        self.meta_processor = meta_processor
        self.is_synthetic = is_synthetic or split_data.get('synthetic', False)
        self.metadata_matrix = meta_processor.transform(self.df)

        self.valid_indices = []
        # Attempt to match IDs using multiple normalization strategies
        for i, row in self.df.iterrows():
            raw_id = str(row['image_fixed']).strip()
            stem_id = Path(raw_id).stem  # Removes .jpg if present
            
            # Check for direct match or stem match
            if raw_id in self.img_map:
                self.valid_indices.append(i)
            elif stem_id in self.img_map:
                # Update the dataframe row to the matching key for later loading
                self.df.at[i, 'image_fixed'] = stem_id
                self.valid_indices.append(i)

        if len(self.valid_indices) == 0:
            logger.error(f"CRITICAL: No images matched for {split_data.get('split', 'unknown')} split!")
            logger.info(f"Sample CSV IDs: {self.df['image'].head(3).tolist()}")
            logger.info(f"Sample Map Keys: {list(self.img_map.keys())[:3]}")

    def __len__(self) -> int:
        return len(self.valid_indices)

    def _load_image(self, img_id: str) -> Optional[np.ndarray]:
        """Load image with comprehensive error handling."""
        img_path = self.img_map.get(img_id)

        if img_path is None or self.is_synthetic:
            # Return a synthetic dermoscopy-like image
            rng = np.random.default_rng(hash(img_id) % 2**32)
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
            # Simulate circular lesion
            center = (self.img_size // 2, self.img_size // 2)
            radius = self.img_size // 3
            color = tuple(int(x) for x in rng.integers(40, 180, 3).tolist())
            cv2.circle(img, center, radius, color, -1)
            # Add noise
            noise = rng.integers(0, 30, img.shape, dtype=np.uint8)
            img = np.clip(img.astype(int) + noise, 0, 255).astype(np.uint8)
            return img

        try:
            img = np.array(Image.open(img_path).convert('RGB'))
            return img
        except Exception as e:
            logger.warning(f"Failed to load {img_path}: {e}")
            return None

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        real_idx = self.valid_indices[idx]
        row = self.df.iloc[real_idx]
        img_id = row['image']
        label = int(row.get('class', 0)) # Ensure this matches your target column 'class'

        # Load and preprocess image
        img = self._load_image(img_id)

        if img is None:
            img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)

        if self.apply_medical_preprocessing and not self.is_synthetic:
            img = preprocessor.full_pipeline(img, output_size=self.img_size)

        # Convert to PIL for torchvision transforms
        pil_img = Image.fromarray(img.astype(np.uint8))

        if self.transform:
            image_tensor = self.transform(pil_img)
        else:
            image_tensor = transforms.ToTensor()(pil_img)

        # Metadata
        meta_vec = torch.tensor(self.metadata_matrix[real_idx], dtype=torch.float32)

        return {
            'image': image_tensor,  # Fix: Return the Tensor, not the ID string
            'metadata': meta_vec,
            'label': torch.tensor(label, dtype=torch.long),
            'image_id': img_id,     # Pass the ID as a separate key if needed for logging
        }


# ============================================================
# SECTION 6: MixUp & CutMix Augmentations
# ============================================================

def mixup_data(x_img, x_meta, y, alpha=0.4):
    """
    MixUp for multimodal data.
    Regularization: prevents overconfident predictions on hard boundaries.
    Clinically: blending borderline cases improves calibration.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    bs = x_img.size(0)
    idx = torch.randperm(bs, device=x_img.device)

    mixed_img = lam * x_img + (1 - lam) * x_img[idx]
    mixed_meta = lam * x_meta + (1 - lam) * x_meta[idx]

    y_a, y_b = y, y[idx]
    return mixed_img, mixed_meta, y_a, y_b, lam


def cutmix_data(x_img, x_meta, y, alpha=1.0):
    """
    CutMix: paste rectangular regions between images.
    More aggressive regularization than MixUp.
    """
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1.0

    bs, C, H, W = x_img.shape
    idx = torch.randperm(bs, device=x_img.device)

    # Cut region
    cut_ratio = np.sqrt(1. - lam)
    cut_w = int(W * cut_ratio)
    cut_h = int(H * cut_ratio)
    cx = np.random.randint(W)
    cy = np.random.randint(H)

    x1 = max(0, cx - cut_w // 2)
    y1 = max(0, cy - cut_h // 2)
    x2 = min(W, cx + cut_w // 2)
    y2 = min(H, cy + cut_h // 2)

    x_img = x_img.clone()
    x_img[:, :, y1:y2, x1:x2] = x_img[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)

    y_a, y_b = y, y[idx]
    return x_img, x_meta, y_a, y_b, lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


# ============================================================
# SECTION 7: Loss Functions
# ============================================================

class FocalLoss(nn.Module):
    """
    Focal Loss for class imbalance.

    Clinical rationale: Melanoma prevalence in ISIC datasets is ~5-15%.
    Standard CE loss dominated by easy non-melanoma negatives → model learns
    to predict 'non-melanoma' for everything, achieving high accuracy but
    dangerously low melanoma sensitivity (false negatives = missed cancers).

    Focal Loss down-weights easy examples (p > 0.5 correct predictions) so
    the model focuses learning on hard, ambiguous lesions — typically the
    early-stage melanomas that are most important to catch.

    gamma=2.0: Standard setting from Lin et al. (2017)
    alpha=0.75: Up-weights melanoma class (positive class)
    """
    def __init__(self, gamma: float = 2.0, alpha: float = 0.75,
                 label_smoothing: float = 0.0, reduction: str = 'mean'):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.label_smoothing = label_smoothing
        self.reduction = reduction

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        # Label smoothing
        if self.label_smoothing > 0:
            n_classes = inputs.size(-1)
            smooth_targets = torch.zeros_like(inputs).scatter_(
                1, targets.unsqueeze(1), 1.0)
            smooth_targets = smooth_targets * (1 - self.label_smoothing) + \
                             self.label_smoothing / n_classes
            log_prob = F.log_softmax(inputs, dim=-1)
            ce = -(smooth_targets * log_prob).sum(dim=-1)
        else:
            ce = F.cross_entropy(inputs, targets, reduction='none')

        p = torch.exp(-ce)
        alpha_t = self.alpha * targets.float() + (1 - self.alpha) * (1 - targets.float())
        focal_loss = alpha_t * (1 - p) ** self.gamma * ce

        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        return focal_loss


# ============================================================
# SECTION 8: SAM Optimizer
# ============================================================

class SAM(torch.optim.Optimizer):
    """
    Sharpness-Aware Minimization (Foret et al., 2021).

    Rationale: Standard optimizers minimize loss at the current point.
    SAM seeks parameters in flat minima — regions where the loss is low
    AND where small perturbations don't increase the loss.

    Medical AI benefit: Flat minima generalize better to out-of-distribution
    dermoscopy images (different devices, imaging conditions, skin tones).
    Sharp minima overfit to training-specific imaging artifacts.
    """
    def __init__(self, params, base_optimizer, rho=0.05, adaptive=False, **kwargs):
        assert rho >= 0.0, "Rho must be non-negative"
        defaults = dict(rho=rho, adaptive=adaptive, **kwargs)
        super(SAM, self).__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group["rho"] / (grad_norm + 1e-12)
            for p in group["params"]:
                if p.grad is None:
                    continue
                self.state[p]["old_p"] = p.data.clone()
                e_w = (torch.pow(p, 2) if group["adaptive"] else 1.0) * p.grad * scale.to(p)
                p.add_(e_w)
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group["params"]:
                if p.grad is None:
                    continue
                p.data = self.state[p]["old_p"]
        self.base_optimizer.step()
        if zero_grad:
            self.zero_grad()

    @torch.no_grad()
    def step(self, closure=None):
        assert closure is not None, "SAM requires closure"
        closure = torch.enable_grad()(closure)
        self.first_step(zero_grad=True)
        closure()
        self.second_step()

    def _grad_norm(self):
        shared_device = self.param_groups[0]["params"][0].device
        norm = torch.norm(
            torch.stack([
                ((torch.abs(p) if group["adaptive"] else 1.0) * p.grad).norm(p=2).to(shared_device)
                for group in self.param_groups for p in group["params"]
                if p.grad is not None
            ]),
            p=2
        )
        return norm


# ============================================================
# SECTION 9: DaViT-Tiny Architecture
# ============================================================
#
# DaViT (Dual Attention Vision Transformer) alternates between:
# - Channel Group Self-Attention: captures global channel correlations
#   (important for capturing cross-pigmentation relationships in melanoma)
# - Spatial Window Self-Attention: captures local spatial patterns
#   (border irregularity, asymmetry detection)
#
# This dual-attention mechanism is particularly powerful for dermoscopy
# because melanoma has BOTH local texture abnormalities (atypical network)
# AND global structure abnormalities (asymmetry, irregular border).

class DropPath(nn.Module):
    """Stochastic Depth: randomly drops entire residual paths during training.
    Reduces co-adaptation of layers, improves generalization."""
    def __init__(self, drop_prob=0.0):
        super().__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        if self.drop_prob == 0. or not self.training:
            return x
        keep_prob = 1 - self.drop_prob
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        random_tensor = torch.rand(shape, dtype=x.dtype, device=x.device)
        random_tensor = torch.floor(random_tensor + keep_prob)
        return x / keep_prob * random_tensor


class ChannelAttention(nn.Module):
    """Channel group self-attention for global feature correlation."""
    def __init__(self, dim, num_heads=8, qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.scale = self.head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)
        
        # Ensure LayerNorm matches the head_dim specifically
        self.norm = nn.LayerNorm(self.head_dim)

    def forward(self, x):
        B, N, C = x.shape
        # qkv: [B, N, 3, num_heads, head_dim]
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim)
        # Permute to [3, B, num_heads, N, head_dim]
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        # Apply normalization to the last dimension (head_dim)
        q = self.norm(q)
        k = self.norm(k)

        # Attention over the spatial dimension N
        # q: [B, num_heads, N, head_dim], k.T: [B, num_heads, head_dim, N]
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        # x: [B, num_heads, N, head_dim] -> [B, N, num_heads, head_dim] -> [B, N, C]
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class WindowAttention(nn.Module):
    """Spatial window self-attention for local feature extraction."""
    def __init__(self, dim, window_size=7, num_heads=8,
                 qkv_bias=True, attn_drop=0., proj_drop=0.):
        super().__init__()
        self.dim = dim
        self.window_size = window_size
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.proj = nn.Linear(dim, dim)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj_drop = nn.Dropout(proj_drop)

        # Relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2*window_size-1) * (2*window_size-1), num_heads))
        nn.init.trunc_normal_(self.relative_position_bias_table, std=0.02)

        coords_h = torch.arange(window_size)
        coords_w = torch.arange(window_size)
        coords = torch.stack(torch.meshgrid([coords_h, coords_w], indexing='ij'))
        coords_flatten = torch.flatten(coords, 1)
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()
        relative_coords[:, :, 0] += window_size - 1
        relative_coords[:, :, 1] += window_size - 1
        relative_coords[:, :, 0] *= 2 * window_size - 1
        relative_position_index = relative_coords.sum(-1)
        self.register_buffer("relative_position_index", relative_position_index)

    def forward(self, x):
        B, N, C = x.shape
        H = W = int(N ** 0.5)

        # Reshape to spatial
        x_2d = x.view(B, H, W, C)
        ws = min(self.window_size, H, W)

        # Pad if needed
        pad_h = (ws - H % ws) % ws
        pad_w = (ws - W % ws) % ws
        if pad_h or pad_w:
            x_2d = F.pad(x_2d, (0, 0, 0, pad_w, 0, pad_h))

        Hp, Wp = x_2d.shape[1], x_2d.shape[2]
        nH, nW = Hp // ws, Wp // ws

        # Window partition
        x_win = x_2d.view(B, nH, ws, nW, ws, C)
        x_win = x_win.permute(0, 1, 3, 2, 4, 5).contiguous()
        x_win = x_win.view(-1, ws*ws, C)

        qkv = self.qkv(x_win).reshape(-1, ws*ws, 3, self.num_heads, C//self.num_heads)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)

        attn = (q @ k.transpose(-2, -1)) * self.scale

        # Add relative position bias
        rel_pos_bias = self.relative_position_bias_table[
            self.relative_position_index[:ws*ws, :ws*ws].reshape(-1)
        ].reshape(ws*ws, ws*ws, -1).permute(2, 0, 1).unsqueeze(0)
        attn = attn + rel_pos_bias

        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x_win = (attn @ v).transpose(1, 2).reshape(-1, ws*ws, C)
        x_win = self.proj(x_win)
        x_win = self.proj_drop(x_win)

        # Reverse window partition
        x_win = x_win.view(B, nH, nW, ws, ws, C)
        x_win = x_win.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, Hp, Wp, C)

        # Remove padding
        if pad_h or pad_w:
            x_win = x_win[:, :H, :W, :].contiguous()

        return x_win.view(B, H*W, C)


class DaViTBlock(nn.Module):
    """Single DaViT block: dual (channel + spatial) attention with MLP."""
    def __init__(self, dim, num_heads=8, window_size=7, mlp_ratio=4.,
                 drop=0., attn_drop=0., drop_path=0., act_layer=nn.GELU):
        super().__init__()
        self.norm1a = nn.LayerNorm(dim)
        self.norm1b = nn.LayerNorm(dim)
        self.channel_attn = ChannelAttention(dim, num_heads, attn_drop=attn_drop, proj_drop=drop)
        self.spatial_attn = WindowAttention(dim, window_size, num_heads,
                                            attn_drop=attn_drop, proj_drop=drop)
        self.norm2 = nn.LayerNorm(dim)
        mlp_hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Linear(dim, mlp_hidden),
            act_layer(),
            nn.Dropout(drop),
            nn.Linear(mlp_hidden, dim),
            nn.Dropout(drop),
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

    def forward(self, x):
        # Channel attention
        x = x + self.drop_path(self.channel_attn(self.norm1a(x)))
        # Spatial attention
        x = x + self.drop_path(self.spatial_attn(self.norm1b(x)))
        # MLP
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x


class DaViTStage(nn.Module):
    """A stage of DaViT blocks with optional downsampling."""
    def __init__(self, in_dim, out_dim, depth, num_heads, window_size=7,
                 downsample=True, drop=0., attn_drop=0., drop_path=0.):
        super().__init__()
        if downsample:
            self.downsample = nn.Sequential(
                nn.LayerNorm(in_dim),
                nn.Linear(in_dim, out_dim),
            )
        else:
            self.downsample = nn.Identity() if in_dim == out_dim else nn.Linear(in_dim, out_dim)

        self.blocks = nn.ModuleList([
            DaViTBlock(
                dim=out_dim, num_heads=num_heads, window_size=window_size,
                drop=drop, attn_drop=attn_drop,
                drop_path=drop_path if isinstance(drop_path, float) else drop_path[i]
            )
            for i in range(depth)
        ])

    def forward(self, x):
        x = self.downsample(x)
        for block in self.blocks:
            x = block(x)
        return x


class DaViTTiny(nn.Module):
    """
    DaViT-Tiny encoder for dermoscopic images.
    Architecture: patch embed → 4 stages of dual attention blocks.
    Output: 768-dim CLS-like feature vector.
    """
    def __init__(self, img_size=224, patch_size=4, in_chans=3,
                 embed_dim=96, depths=(1, 1, 3, 1), num_heads=(3, 6, 12, 24),
                 window_size=7, drop_rate=0., attn_drop_rate=0.,
                 stoch_depth_rate=0.1):
        super().__init__()
        self.patch_size = patch_size
        self.embed_dim = embed_dim

        # Patch embedding
        self.patch_embed = nn.Sequential(
            nn.Conv2d(in_chans, embed_dim, kernel_size=7, stride=4, padding=3),
            nn.LayerNorm([embed_dim,
                          img_size // patch_size,
                          img_size // patch_size])
        )

        # Stochastic depth decay
        total_depth = sum(depths)
        dpr = [x.item() for x in torch.linspace(0, stoch_depth_rate, total_depth)]

        # Build stages
        dims = [embed_dim * (2**i) for i in range(4)]
        self.stages = nn.ModuleList()
        block_idx = 0
        for i, (depth, heads) in enumerate(zip(depths, num_heads)):
            in_dim = dims[i-1] if i > 0 else dims[0]
            out_dim = dims[i]
            stage_dpr = dpr[block_idx:block_idx + depth]
            self.stages.append(DaViTStage(
                in_dim=in_dim, out_dim=out_dim, depth=depth, num_heads=heads,
                window_size=window_size, downsample=(i > 0),
                drop=drop_rate, attn_drop=attn_drop_rate,
                drop_path=stage_dpr if depth > 1 else stage_dpr[0]
            ))
            block_idx += depth

        self.norm = nn.LayerNorm(dims[-1])
        self.out_dim = dims[-1]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, C, H, W = x.shape

        # Patch embed: B, embed_dim, H/4, W/4
        x = self.patch_embed(x)

        # Reshape to sequence
        B, D, H, W = x.shape
        x = x.permute(0, 2, 3, 1).reshape(B, H*W, D)

        # Downsampling between stages
        for i, stage in enumerate(self.stages):
            if i > 0:
                # Spatial downsampling (2x) via reshape + linear
                H_new, W_new = H // 2, W // 2
                x_2d = x.view(B, H, W, -1)
                # Average 2x2 windows
                x_2d = x_2d.view(B, H//2, 2, W//2, 2, -1).mean(dim=[2, 4])
                x = x_2d.reshape(B, H_new*W_new, -1)
                H, W = H_new, W_new
            x = stage(x)

        x = self.norm(x)
        # Global average pooling
        x = x.mean(dim=1)  # B, out_dim
        return x


# ============================================================
# SECTION 10: Metadata Encoder
# ============================================================

class MetadataEncoder(nn.Module):
    """
    Deep metadata encoder with residual connections.
    Transforms clinical features into a rich embedding.
    """
    def __init__(self, input_dim: int, embed_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, embed_dim),
            nn.LayerNorm(embed_dim),
            nn.GELU(),
            nn.Dropout(dropout // 2),
        )
        # Residual projection if dims differ
        self.residual = nn.Linear(input_dim, embed_dim) if input_dim != embed_dim else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x) + self.residual(x) if x.shape[-1] != self.encoder[0].out_features else self.encoder(x)


# ============================================================
# SECTION 11: Gated Fusion Module
# ============================================================

class GatedFusion(nn.Module):
    """
    Learned Gated Fusion for image + metadata.

    Architecture:
        gate_img  = sigmoid(W_g_img  * [f_img  || f_meta])
        gate_meta = sigmoid(W_g_meta * [f_img  || f_meta])
        f_fused   = gate_img * f_img + gate_meta * f_meta
        f_out     = MLP(f_fused)

    Clinical rationale:
    Metadata quality is heterogeneous in real clinical settings:
    - Some patients have complete, accurate clinical records → metadata should
      be highly weighted
    - Others have missing age, unconfirmed diagnosis, unclear anatomical site →
      metadata should be down-weighted in favor of image evidence

    Gated Fusion learns to modulate this balance from the data itself.
    The sigmoid gates produce values in [0,1] representing how much to trust
    each modality for each specific patient — an implicit clinical confidence score.

    Gate activations can also be used for MULTIMODAL XAI: high gate_meta → the
    model relied heavily on clinical features for this prediction.
    """
    def __init__(self, img_dim: int, meta_dim: int, fusion_dim: int, dropout: float = 0.3):
        super().__init__()
        combined_dim = img_dim + meta_dim

        # Gate networks
        self.gate_img = nn.Sequential(
            nn.Linear(combined_dim, combined_dim // 2),
            nn.GELU(),
            nn.Linear(combined_dim // 2, img_dim),
            nn.Sigmoid()
        )
        self.gate_meta = nn.Sequential(
            nn.Linear(combined_dim, combined_dim // 2),
            nn.GELU(),
            nn.Linear(combined_dim // 2, meta_dim),
            nn.Sigmoid()
        )

        # Projection to common fusion space
        self.img_proj = nn.Linear(img_dim, fusion_dim)
        self.meta_proj = nn.Linear(meta_dim, fusion_dim)

        # Fusion MLP
        self.fusion_mlp = nn.Sequential(
            nn.LayerNorm(fusion_dim),
            nn.Linear(fusion_dim, fusion_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, f_img: torch.Tensor, f_meta: torch.Tensor
                ) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        combined = torch.cat([f_img, f_meta], dim=-1)

        # Compute gates
        g_img = self.gate_img(combined)   # B, img_dim
        g_meta = self.gate_meta(combined) # B, meta_dim

        # Gated features
        f_img_gated = g_img * f_img
        f_meta_gated = g_meta * f_meta

        # Project to fusion space
        f_img_proj = self.img_proj(f_img_gated)    # B, fusion_dim
        f_meta_proj = self.meta_proj(f_meta_gated) # B, fusion_dim

        # Element-wise sum (preserves gradients to both branches)
        f_fused = f_img_proj + f_meta_proj
        f_out = self.fusion_mlp(f_fused)

        # Return gate values for XAI analysis
        gate_values = {
            'gate_img': g_img.detach(),
            'gate_meta': g_meta.detach(),
            'img_contribution': f_img_proj.detach().norm(dim=-1),
            'meta_contribution': f_meta_proj.detach().norm(dim=-1),
        }

        return f_out, gate_values


# ============================================================
# SECTION 12: Full DaViT + Gated Fusion Model
# ============================================================

class DaViTGatedFusionModel(nn.Module):
    """
    Complete multimodal model: DaViT-Tiny image encoder + Gated Fusion.
    Optimized for speed + accuracy with MC Dropout support.
    """
    def __init__(self, metadata_input_dim: int, cfg: Dict):
        super().__init__()

        # Image encoder
        self.image_encoder = DaViTTiny(
            img_size=cfg['img_size'],
            embed_dim=cfg['embed_dim'],
            depths=cfg['depths'],
            num_heads=cfg['num_heads'],
            stoch_depth_rate=cfg['stoch_depth_rate'],
            drop_rate=cfg['dropout'] * 0.5,
            attn_drop_rate=cfg['dropout'] * 0.5,
        )
        self.image_encoder.gradient_checkpointing = True
        img_feature_dim = self.image_encoder.out_dim

        # Metadata encoder
        self.metadata_encoder = MetadataEncoder(
            input_dim=metadata_input_dim,
            embed_dim=cfg['metadata_embed_dim'],
            dropout=cfg['dropout'],
        )

        # Gated fusion
        self.fusion = GatedFusion(
            img_dim=img_feature_dim,
            meta_dim=cfg['metadata_embed_dim'],
            fusion_dim=cfg['fusion_dim'],
            dropout=cfg['dropout'],
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(cfg['dropout']),
            nn.Linear(cfg['fusion_dim'], cfg['fusion_dim'] // 2),
            nn.GELU(),
            nn.Dropout(cfg['dropout'] * 0.5),
            nn.Linear(cfg['fusion_dim'] // 2, cfg['num_classes']),
        )

        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.LayerNorm):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, image: torch.Tensor, metadata: torch.Tensor,
                return_gates: bool = False
                ) -> Union[torch.Tensor, Tuple[torch.Tensor, Dict]]:

        # Encode image
        f_img = self.image_encoder(image)

        # Encode metadata
        f_meta = self.metadata_encoder(metadata)

        # Gated fusion
        f_fused, gate_values = self.fusion(f_img, f_meta)

        # Classification
        logits = self.classifier(f_fused)

        if return_gates:
            return logits, gate_values
        return logits

    def enable_mc_dropout(self):
        """Enable dropout layers for MC dropout inference."""
        for m in self.modules():
            if isinstance(m, nn.Dropout):
                m.train()

    def get_features(self, image: torch.Tensor, metadata: torch.Tensor) -> torch.Tensor:
        """Extract fused features (for t-SNE, UMAP)."""
        f_img = self.image_encoder(image)
        f_meta = self.metadata_encoder(metadata)
        f_fused, _ = self.fusion(f_img, f_meta)
        return f_fused


# ============================================================
# SECTION 13: EMA (Exponential Moving Average)
# ============================================================

class EMA:
    """
    Exponential Moving Average of model weights.
    Creates a smoothed version of the model that generalizes better.
    Used for final inference and evaluation.
    """
    def __init__(self, model: nn.Module, decay: float = 0.9998):
        self.decay = decay
        self.shadow = deepcopy(model).eval()
        for p in self.shadow.parameters():
            p.requires_grad_(False)

    @torch.no_grad()
    def update(self, model: nn.Module):
        for shadow_p, model_p in zip(self.shadow.parameters(), model.parameters()):
            shadow_p.data = self.decay * shadow_p.data + (1 - self.decay) * model_p.data

    def __call__(self, *args, **kwargs):
        return self.shadow(*args, **kwargs)


# ============================================================
# SECTION 14: Training Pipeline
# ============================================================

class WarmupCosineScheduler:
    """Linear warmup + cosine annealing LR scheduler."""
    def __init__(self, optimizer, warmup_epochs, total_epochs, min_lr=1e-6):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.total_epochs = total_epochs
        self.min_lr = min_lr
        self.base_lrs = [pg['lr'] for pg in optimizer.param_groups]

    def step(self, epoch):
        if epoch < self.warmup_epochs:
            scale = (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / (self.total_epochs - self.warmup_epochs)
            scale = self.min_lr / self.base_lrs[0] + \
                    0.5 * (1 - self.min_lr / self.base_lrs[0]) * \
                    (1 + math.cos(math.pi * progress))

        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg['lr'] = base_lr * scale

    def get_lr(self):
        return [pg['lr'] for pg in self.optimizer.param_groups][0]


class Trainer:
    """
    Full training pipeline with:
    - AMP, SAM, EMA, gradient clipping
    - MixUp/CutMix
    - Early stopping
    - Comprehensive logging
    """

    def __init__(self, model: nn.Module, cfg: Dict, device: torch.device):
        self.model = model.to(device)
        self.cfg = cfg
        self.device = device
        self.ema = EMA(model, decay=cfg['ema_decay'])
        self.scaler = GradScaler()

        # Loss
        self.criterion = FocalLoss(
            gamma=cfg['focal_gamma'],
            alpha=cfg['focal_alpha'],
            label_smoothing=cfg['label_smoothing']
        )

        # Optimizer
        param_groups = self._get_param_groups()
        if cfg['use_sam']:
            self.optimizer = SAM(param_groups, torch.optim.AdamW,
                                 rho=cfg['sam_rho'],
                                 lr=cfg['lr'],
                                 weight_decay=cfg['weight_decay'])
        else:
            self.optimizer = torch.optim.AdamW(
                param_groups, lr=cfg['lr'], weight_decay=cfg['weight_decay'])

        self.scheduler = WarmupCosineScheduler(
            self.optimizer if not cfg['use_sam'] else self.optimizer.base_optimizer,
            warmup_epochs=cfg['warmup_epochs'],
            total_epochs=cfg['epochs'],
            min_lr=cfg['min_lr']
        )

        self.history = {'train_loss': [], 'val_loss': [], 'val_auc': [],
                        'train_auc': [], 'lr': []}
        self.best_val_auc = 0.0
        self.patience_counter = 0

    def _get_param_groups(self):
        """Layer-wise LR decay for pre-trained layers."""
        no_decay = ['bias', 'LayerNorm', 'norm']
        return [
            {'params': [p for n, p in self.model.named_parameters()
                        if not any(nd in n for nd in no_decay) and 'image_encoder' in n],
             'weight_decay': self.cfg['weight_decay'], 'lr': self.cfg['lr'] * 0.1},
            {'params': [p for n, p in self.model.named_parameters()
                        if any(nd in n for nd in no_decay) and 'image_encoder' in n],
             'weight_decay': 0.0, 'lr': self.cfg['lr'] * 0.1},
            {'params': [p for n, p in self.model.named_parameters()
                        if 'image_encoder' not in n],
             'weight_decay': self.cfg['weight_decay'], 'lr': self.cfg['lr']},
        ]

    def train_epoch(self, loader: DataLoader, epoch: int) -> Dict:
        self.model.train()
        total_loss = 0.0
        all_preds, all_labels = [], []

        for step, batch in enumerate(loader):
            images = batch['image'].to(self.device)
            metadata = batch['metadata'].to(self.device)
            labels = batch['label'].to(self.device)

            # MixUp or CutMix
            use_mix = random.random() < self.cfg['mixup_prob']
            mixed_imgs, mixed_meta, y_a, y_b, lam = images, metadata, labels, labels, 1.0

            if use_mix and self.model.training:
                if random.random() < 0.5:
                    mixed_imgs, mixed_meta, y_a, y_b, lam = mixup_data(
                        images, metadata, labels, self.cfg['mixup_alpha'])
                else:
                    mixed_imgs, mixed_meta, y_a, y_b, lam = cutmix_data(
                        images, metadata, labels, self.cfg['cutmix_alpha'])

            if self.cfg['use_sam']:
                # SAM: first step
                with autocast():
                    logits = self.model(mixed_imgs, mixed_meta)
                    loss = mixup_criterion(self.criterion, logits, y_a, y_b, lam) \
                           if use_mix else self.criterion(logits, labels)
                    loss = loss / self.cfg['grad_accum_steps']
                self.scaler.scale(loss).backward()

                if (step + 1) % self.cfg['grad_accum_steps'] == 0:
                    self.scaler.unscale_(self.optimizer.base_optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg['clip_grad'])
                    self.scaler.step(self.optimizer.base_optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()

                    # SAM second step
                    self.optimizer.first_step(zero_grad=True)
                    with autocast():
                        logits2 = self.model(mixed_imgs, mixed_meta)
                        loss2 = mixup_criterion(self.criterion, logits2, y_a, y_b, lam) \
                                if use_mix else self.criterion(logits2, labels)
                    self.scaler.scale(loss2).backward()
                    self.scaler.unscale_(self.optimizer.base_optimizer)
                    self.optimizer.second_step(zero_grad=True)
                    self.scaler.update()
            else:
                with autocast():
                    logits = self.model(mixed_imgs, mixed_meta)
                    loss = mixup_criterion(self.criterion, logits, y_a, y_b, lam) \
                           if use_mix else self.criterion(logits, labels)
                    loss = loss / self.cfg['grad_accum_steps']

                self.scaler.scale(loss).backward()

                if (step + 1) % self.cfg['grad_accum_steps'] == 0:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), self.cfg['clip_grad'])
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                    self.optimizer.zero_grad()

            self.ema.update(self.model)
            total_loss += loss.item() * self.cfg['grad_accum_steps']

            with torch.no_grad():
                probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
                all_preds.extend(probs.tolist())
                all_labels.extend(labels.cpu().numpy().tolist())

        avg_loss = total_loss / len(loader)
        auc = roc_auc_score(all_labels, all_preds) if len(set(all_labels)) > 1 else 0.5
        return {'loss': avg_loss, 'auc': auc}

    @torch.no_grad()
    def evaluate(self, loader: DataLoader, use_ema: bool = True) -> Dict:
        model = self.ema.shadow if use_ema else self.model
        model.eval()
        total_loss = 0.0
        all_probs, all_labels = [], []

        for batch in loader:
            images = batch['image'].to(self.device)
            metadata = batch['metadata'].to(self.device)
            labels = batch['label'].to(self.device)

            with autocast():
                logits = model(images, metadata)
                loss = self.criterion(logits, labels)

            probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs.tolist())
            all_labels.extend(labels.cpu().numpy().tolist())
            total_loss += loss.item()

        all_probs = np.array(all_probs)
        all_labels = np.array(all_labels)
        preds = (all_probs >= 0.5).astype(int)

        metrics = {
            'loss': total_loss / len(loader),
            'auc': roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else 0.5,
            'pr_auc': average_precision_score(all_labels, all_probs),
            'accuracy': accuracy_score(all_labels, preds),
            'recall': recall_score(all_labels, preds, zero_division=0),
            'precision': precision_score(all_labels, preds, zero_division=0),
            'f1': f1_score(all_labels, preds, zero_division=0),
            'brier': brier_score_loss(all_labels, all_probs),
            'probs': all_probs,
            'labels': all_labels,
            'preds': preds,
        }

        # Sensitivity / Specificity
        cm = confusion_matrix(all_labels, preds)
        if cm.shape == (2, 2):
            tn, fp, fn, tp = cm.ravel()
            metrics['sensitivity'] = tp / (tp + fn + 1e-8)
            metrics['specificity'] = tn / (tn + fp + 1e-8)
            metrics['ppv'] = tp / (tp + fp + 1e-8)
            metrics['npv'] = tn / (tn + fn + 1e-8)
            metrics['fnr'] = fn / (fn + tp + 1e-8)
            metrics['youden'] = metrics['sensitivity'] + metrics['specificity'] - 1

        return metrics

    def fit(self, train_loader: DataLoader, val_loader: DataLoader) -> Dict:
        logger.info("=" * 70)
        logger.info("TRAINING STARTED: DaViT-Tiny + Gated Fusion")
        logger.info("=" * 70)

        for epoch in range(self.cfg['epochs']):
            t0 = time.time()
            self.scheduler.step(epoch)

            train_metrics = self.train_epoch(train_loader, epoch)
            val_metrics = self.evaluate(val_loader, use_ema=True)

            lr = self.scheduler.get_lr()
            elapsed = time.time() - t0

            self.history['train_loss'].append(train_metrics['loss'])
            self.history['train_auc'].append(train_metrics['auc'])
            self.history['val_loss'].append(val_metrics['loss'])
            self.history['val_auc'].append(val_metrics['auc'])
            self.history['lr'].append(lr)

            logger.info(
                f"Epoch {epoch+1:3d}/{self.cfg['epochs']} | "
                f"Train Loss: {train_metrics['loss']:.4f} AUC: {train_metrics['auc']:.4f} | "
                f"Val Loss: {val_metrics['loss']:.4f} AUC: {val_metrics['auc']:.4f} | "
                f"Sens: {val_metrics.get('sensitivity', 0):.4f} | "
                f"LR: {lr:.6f} | Time: {elapsed:.1f}s"
            )

            # Early stopping
            if val_metrics['auc'] > self.best_val_auc:
                self.best_val_auc = val_metrics['auc']
                self.patience_counter = 0
                # Save checkpoint
                torch.save({
                    'epoch': epoch,
                    'model_state': self.model.state_dict(),
                    'ema_state': self.ema.shadow.state_dict(),
                    'optimizer_state': self.optimizer.state_dict() if not self.cfg['use_sam']
                                       else self.optimizer.base_optimizer.state_dict(),
                    'val_auc': self.best_val_auc,
                    'cfg': self.cfg,
                }, f"{self.cfg['output_dir']}/checkpoints/best_model.pth")
                logger.info(f"  → New best AUC: {self.best_val_auc:.4f} (saved)")
            else:
                self.patience_counter += 1
                if self.patience_counter >= self.cfg['patience']:
                    logger.info(f"Early stopping at epoch {epoch+1}")
                    break

        return self.history


# ============================================================
# SECTION 15: Comprehensive Evaluation
# ============================================================

def compute_full_metrics(probs: np.ndarray, labels: np.ndarray,
                         threshold: float = 0.5) -> Dict:
    """Compute all medical AI metrics."""
    preds = (probs >= threshold).astype(int)
    cm = confusion_matrix(labels, preds)

    metrics = {
        'accuracy': accuracy_score(labels, preds),
        'precision': precision_score(labels, preds, zero_division=0),
        'recall': recall_score(labels, preds, zero_division=0),
        'f1': f1_score(labels, preds, zero_division=0),
        'roc_auc': roc_auc_score(labels, probs) if len(set(labels)) > 1 else 0.5,
        'pr_auc': average_precision_score(labels, probs),
        'brier': brier_score_loss(labels, probs),
        'confusion_matrix': cm,
    }

    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        metrics['sensitivity'] = tp / (tp + fn + 1e-8)
        metrics['specificity'] = tn / (tn + fp + 1e-8)
        metrics['ppv'] = tp / (tp + fp + 1e-8)
        metrics['npv'] = tn / (tn + fn + 1e-8)
        metrics['fnr'] = fn / (fn + tp + 1e-8)
        metrics['fpr'] = fp / (fp + tn + 1e-8)
        metrics['youden'] = metrics['sensitivity'] + metrics['specificity'] - 1
        metrics['tp'] = int(tp)
        metrics['fp'] = int(fp)
        metrics['tn'] = int(tn)
        metrics['fn'] = int(fn)

    # Sensitivity at fixed specificity levels
    fpr_arr, tpr_arr, thresholds = roc_curve(labels, probs)
    for target_spec in [0.80, 0.85, 0.90, 0.95]:
        target_fpr = 1 - target_spec
        idx = np.argmin(np.abs(fpr_arr - target_fpr))
        metrics[f'sensitivity_at_spec{int(target_spec*100)}'] = tpr_arr[idx]

    # Youden optimal threshold
    youden_idx = np.argmax(tpr_arr - fpr_arr)
    metrics['youden_threshold'] = thresholds[youden_idx]

    return metrics


def print_metrics_table(metrics: Dict, title: str = "EVALUATION RESULTS"):
    """Print publication-style metrics table."""
    print("\n" + "=" * 60)
    print(f"  {title}")
    print("=" * 60)
    keys = ['accuracy', 'roc_auc', 'pr_auc', 'sensitivity', 'specificity',
            'ppv', 'npv', 'f1', 'brier', 'fnr', 'youden']
    for k in keys:
        if k in metrics:
            print(f"  {k:<35} {metrics[k]:.4f}")
    print("=" * 60)
    for s in [80, 85, 90, 95]:
        k = f'sensitivity_at_spec{s}'
        if k in metrics:
            print(f"  Sensitivity @ Specificity={s}%:  {metrics[k]:.4f}")
    print("=" * 60)


# ============================================================
# SECTION 16: Temperature Scaling (Calibration)
# ============================================================
#
# WHY CALIBRATION MATTERS CLINICALLY:
# An uncalibrated model might predict 95% probability of melanoma when it
# should predict 60%. This overconfidence causes clinicians to:
# (1) Over-trust borderline cases → unnecessary biopsies (patient harm)
# (2) Reduce vigilance on lower-probability cases → missed melanoma
# Temperature scaling is a simple, post-hoc calibration method that
# scales logits by a single learned parameter T: p = softmax(logits/T)
# T > 1 → softer probabilities (reduces overconfidence)
# T < 1 → sharper probabilities (rarely needed)

class TemperatureScaling(nn.Module):
    """Post-hoc temperature scaling for probability calibration."""
    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return logits / self.temperature

    def fit(self, logits: torch.Tensor, labels: torch.Tensor,
            n_iter: int = 100, lr: float = 0.01) -> float:
        """Optimize temperature on validation set."""
        optimizer = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=n_iter)
        nll_criterion = nn.CrossEntropyLoss()

        def eval_fn():
            optimizer.zero_grad()
            scaled = self.forward(logits)
            loss = nll_criterion(scaled, labels)
            loss.backward()
            return loss

        optimizer.step(eval_fn)
        logger.info(f"Temperature scaled: T = {self.temperature.item():.4f}")
        return self.temperature.item()


def compute_ece(probs: np.ndarray, labels: np.ndarray, n_bins: int = 15) -> float:
    """Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (probs >= bins[i]) & (probs < bins[i+1])
        if mask.sum() == 0:
            continue
        avg_conf = probs[mask].mean()
        avg_acc = labels[mask].mean()
        ece += mask.sum() / len(labels) * np.abs(avg_conf - avg_acc)
    return ece


# ============================================================
# SECTION 17: Monte Carlo Dropout Uncertainty Estimation
# ============================================================
#
# MC Dropout approximates Bayesian inference:
# By running forward passes with dropout active (at test time),
# we sample from the approximate posterior over model weights.
# Variance across T samples = epistemic uncertainty (model doesn't know).
# This distinguishes:
# - Aleatoric uncertainty (inherent ambiguity in the lesion)
# - Epistemic uncertainty (model hasn't seen enough similar cases)
# High uncertainty predictions should be flagged for dermatologist review.

@torch.no_grad()
def mc_dropout_predict(model: nn.Module, batch: Dict,
                       n_samples: int = 30, device: torch.device = DEVICE) -> Dict:
    """MC Dropout inference: multiple stochastic forward passes."""
    model.eval()
    model.enable_mc_dropout()

    images = batch['image'].to(device)
    metadata = batch['metadata'].to(device)

    all_probs = []
    for _ in range(n_samples):
        logits = model(images, metadata)
        probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        all_probs.append(probs)

    all_probs = np.stack(all_probs, axis=0)  # T, B

    mean_probs = all_probs.mean(axis=0)
    std_probs = all_probs.std(axis=0)
    pred_entropy = -mean_probs * np.log(mean_probs + 1e-8) - \
                   (1 - mean_probs) * np.log(1 - mean_probs + 1e-8)

    return {
        'mean_probs': mean_probs,
        'std_probs': std_probs,
        'entropy': pred_entropy,
        'all_probs': all_probs,
    }


# ============================================================
# SECTION 18: Test-Time Augmentation (TTA)
# ============================================================

@torch.no_grad()
def tta_predict(model: nn.Module, dataset: ISICMultimodalDataset,
                tta_transforms: List, device: torch.device = DEVICE,
                batch_size: int = 32) -> np.ndarray:
    """TTA: average predictions across multiple augmented views."""
    model.eval()
    all_tta_probs = []

    for tta_transform in tta_transforms:
        dataset_copy = deepcopy(dataset)
        dataset_copy.transform = tta_transform
        loader = DataLoader(dataset_copy, batch_size=batch_size,
                            shuffle=False, num_workers=0)

        probs_this_aug = []
        for batch in loader:
            images = batch['image'].to(device)
            metadata = batch['metadata'].to(device)
            logits = model(images, metadata)
            probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            probs_this_aug.extend(probs.tolist())

        all_tta_probs.append(probs_this_aug)

    return np.mean(all_tta_probs, axis=0)


# ============================================================
# SECTION 19: XAI — Grad-CAM Family
# ============================================================

class GradCAM:
    """Gradient-weighted Class Activation Maps for DaViT."""

    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._hooks = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()

        def backward_hook(module, grad_in, grad_out):
            self.gradients = grad_out[0].detach()

        self._hooks.append(self.target_layer.register_forward_hook(forward_hook))
        self._hooks.append(self.target_layer.register_full_backward_hook(backward_hook))

    def __call__(self, image: torch.Tensor, metadata: torch.Tensor,
                 target_class: int = 1) -> np.ndarray:
        self.model.eval()

        output = self.model(image, metadata)
        self.model.zero_grad()
        output[0, target_class].backward()

        if self.gradients is None or self.activations is None:
            return np.zeros((image.shape[2], image.shape[3]))

        # Handle both 2D (spatial) and 1D (sequence) activations
        acts = self.activations
        grads = self.gradients

        if acts.ndim == 3:  # B, N, C (sequence)
            weights = grads.mean(dim=1)  # B, C
            cam = (weights.unsqueeze(1) * acts).sum(dim=-1)  # B, N
            N = cam.shape[1]
            H = W = int(N ** 0.5)
            if H * W == N:
                cam = cam[0].view(H, W).cpu().numpy()
            else:
                cam = cam[0].cpu().numpy()
                H = W = int(np.sqrt(N))
                cam = cam[:H*W].reshape(H, W)
        elif acts.ndim == 4:  # B, C, H, W
            weights = grads.mean(dim=[2, 3])  # B, C
            cam = (weights[:, :, None, None] * acts).sum(dim=1)[0].cpu().numpy()

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (image.shape[3], image.shape[2]))
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam

    def remove_hooks(self):
        for h in self._hooks:
            h.remove()


class GradCAMPlusPlus(GradCAM):
    """Grad-CAM++ with improved localization."""

    def __call__(self, image: torch.Tensor, metadata: torch.Tensor,
                 target_class: int = 1) -> np.ndarray:
        self.model.eval()
        output = self.model(image, metadata)
        self.model.zero_grad()
        output[0, target_class].backward()

        if self.gradients is None or self.activations is None:
            return np.zeros((image.shape[2], image.shape[3]))

        acts = self.activations
        grads = self.gradients

        if acts.ndim == 4:  # B, C, H, W
            # Grad-CAM++ weight computation
            alpha_num = grads ** 2
            alpha_denom = 2 * grads**2 + (acts * grads**3).sum(dim=[2,3], keepdim=True) + 1e-7
            alpha = alpha_num / alpha_denom
            weights = (alpha * F.relu(grads)).sum(dim=[2, 3])  # B, C
            cam = (weights[:, :, None, None] * acts).sum(dim=1)[0].cpu().numpy()
        else:
            # Sequence format fallback
            weights = grads.mean(dim=1)
            cam = (weights.unsqueeze(1) * acts).sum(dim=-1)[0].cpu().numpy()
            N = cam.shape[0]
            H = W = int(N ** 0.5)
            cam = cam[:H*W].reshape(H, W)

        cam = np.maximum(cam, 0)
        cam = cv2.resize(cam, (image.shape[3], image.shape[2]))
        if cam.max() > cam.min():
            cam = (cam - cam.min()) / (cam.max() - cam.min())
        return cam


class OcclusionSensitivity:
    """
    Occlusion Sensitivity: slide a mask over the image, measure prediction drop.
    More model-agnostic than gradient methods.
    """
    def __init__(self, model: nn.Module, patch_size: int = 32, stride: int = 16):
        self.model = model
        self.patch_size = patch_size
        self.stride = stride

    @torch.no_grad()
    def __call__(self, image: torch.Tensor, metadata: torch.Tensor,
                 target_class: int = 1) -> np.ndarray:
        self.model.eval()
        B, C, H, W = image.shape

        # Baseline prediction
        base_logit = F.softmax(self.model(image, metadata), dim=-1)[0, target_class].item()

        heatmap = np.zeros((H, W))
        count = np.zeros((H, W))

        for y in range(0, H - self.patch_size + 1, self.stride):
            for x in range(0, W - self.patch_size + 1, self.stride):
                occluded = image.clone()
                occluded[:, :, y:y+self.patch_size, x:x+self.patch_size] = 0.0

                pred = F.softmax(self.model(occluded, metadata), dim=-1)[0, target_class].item()
                drop = base_logit - pred  # positive = important region

                heatmap[y:y+self.patch_size, x:x+self.patch_size] += drop
                count[y:y+self.patch_size, x:x+self.patch_size] += 1

        count = np.maximum(count, 1)
        heatmap = heatmap / count
        if heatmap.max() > heatmap.min():
            heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min())
        return heatmap


class IntegratedGradients:
    """
    Integrated Gradients: attribute predictions to input features
    via path integral from baseline to input.

    Baseline choice: Mean image (not black/zero — avoids spurious attributions
    from out-of-distribution inputs).
    """
    def __init__(self, model: nn.Module, n_steps: int = 50):
        self.model = model
        self.n_steps = n_steps

    def __call__(self, image: torch.Tensor, metadata: torch.Tensor,
                 target_class: int = 1, baseline: Optional[torch.Tensor] = None) -> np.ndarray:
        if baseline is None:
            baseline = image.mean(dim=[2, 3], keepdim=True).expand_as(image)

        alphas = torch.linspace(0, 1, self.n_steps, device=image.device)
        integrated_grads = torch.zeros_like(image)

        for alpha in alphas:
            interpolated = baseline + alpha * (image - baseline)
            interpolated.requires_grad_(True)

            output = self.model(interpolated, metadata)
            self.model.zero_grad()
            output[0, target_class].backward()

            if interpolated.grad is not None:
                integrated_grads += interpolated.grad.detach()

        # Average and multiply by (input - baseline)
        ig = (image - baseline) * integrated_grads / self.n_steps
        ig_attr = ig[0].abs().mean(dim=0).cpu().numpy()  # Average across channels

        if ig_attr.max() > ig_attr.min():
            ig_attr = (ig_attr - ig_attr.min()) / (ig_attr.max() - ig_attr.min())
        return ig_attr


class SmoothGrad:
    """
    SmoothGrad: average gradients over noisy input copies.
    Reduces noise in gradient maps for cleaner explanations.
    """
    def __init__(self, model: nn.Module, n_samples: int = 50,
                 noise_level: float = 0.15):
        self.model = model
        self.n_samples = n_samples
        self.noise_level = noise_level

    def __call__(self, image: torch.Tensor, metadata: torch.Tensor,
                 target_class: int = 1) -> np.ndarray:
        noise_std = self.noise_level * (image.max() - image.min()).item()
        grad_sum = torch.zeros_like(image)

        for _ in range(self.n_samples):
            noisy = image + torch.randn_like(image) * noise_std
            noisy.requires_grad_(True)

            output = self.model(noisy, metadata)
            self.model.zero_grad()
            output[0, target_class].backward()

            if noisy.grad is not None:
                grad_sum += noisy.grad.detach()

        avg_grad = (grad_sum / self.n_samples)[0].abs().mean(dim=0).cpu().numpy()
        if avg_grad.max() > avg_grad.min():
            avg_grad = (avg_grad - avg_grad.min()) / (avg_grad.max() - avg_grad.min())
        return avg_grad


# ============================================================
# SECTION 20: Quantitative XAI Evaluation
# ============================================================
#
# CRITICAL: Qualitative saliency maps are insufficient.
# We must quantitatively verify that highlighted regions are
# causally important for the prediction.
#
# Insertion: progressively reveal important pixels → AUC should rise fast
# Deletion: progressively remove important pixels → AUC should drop fast
# If explanations are faithful, insertion AUC > deletion AUC significantly.

@torch.no_grad()
def insertion_deletion_metrics(model: nn.Module, image: torch.Tensor,
                                metadata: torch.Tensor, saliency_map: np.ndarray,
                                n_steps: int = 100, device: torch.device = DEVICE,
                                blur_baseline: bool = True) -> Dict:
    """
    Compute Insertion and Deletion metrics for XAI faithfulness evaluation.

    Insertion: start from blurred baseline, progressively insert most important pixels.
    Deletion: start from original image, progressively delete most important pixels.

    Higher insertion AUC = better faithfulness.
    Lower deletion AUC = better faithfulness.
    """
    model.eval()
    image = image.to(device)
    metadata = metadata.to(device)

    # Create baseline (blurred — avoids out-of-distribution black image)
    if blur_baseline:
        img_np = image[0].cpu().permute(1, 2, 0).numpy()
        img_np = np.clip(img_np * np.array([0.229, 0.224, 0.225]) +
                         np.array([0.485, 0.456, 0.406]), 0, 1)
        img_blurred = cv2.GaussianBlur((img_np * 255).astype(np.uint8), (51, 51), 10)
        # Re-normalize
        img_blurred = (img_blurred / 255.0 - np.array([0.485, 0.456, 0.406])) / \
                      np.array([0.229, 0.224, 0.225])
        baseline = torch.tensor(img_blurred, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0).to(device)
    else:
        baseline = torch.zeros_like(image)

    # Sort pixels by importance (descending)
    flat_sal = saliency_map.flatten()
    H, W = saliency_map.shape
    sorted_idx = np.argsort(-flat_sal)
    n_pixels = len(sorted_idx)
    step_size = max(1, n_pixels // n_steps)

    insertion_scores = []
    deletion_scores = []

    for i in range(0, n_pixels, step_size):
        n_revealed = min(i + step_size, n_pixels)
        mask = np.zeros(n_pixels, dtype=np.float32)
        mask[sorted_idx[:n_revealed]] = 1.0
        mask_2d = mask.reshape(H, W)
        mask_tensor = torch.tensor(mask_2d, dtype=torch.float32).to(device)[None, None]

        # Insertion: baseline + revealed pixels from original
        ins_img = baseline * (1 - mask_tensor) + image * mask_tensor
        ins_pred = F.softmax(model(ins_img, metadata), dim=-1)[0, 1].item()
        insertion_scores.append(ins_pred)

        # Deletion: original - important pixels → replaced with baseline
        del_img = image * (1 - mask_tensor) + baseline * mask_tensor
        del_pred = F.softmax(model(del_img, metadata), dim=-1)[0, 1].item()
        deletion_scores.append(del_pred)

    insertion_auc = np.trapz(insertion_scores) / len(insertion_scores)
    deletion_auc = np.trapz(deletion_scores) / len(deletion_scores)

    return {
        'insertion_auc': insertion_auc,
        'deletion_auc': deletion_auc,
        'insertion_scores': insertion_scores,
        'deletion_scores': deletion_scores,
        'faithfulness_gap': insertion_auc - deletion_auc,
    }


def average_drop_increase(model: nn.Module, image: torch.Tensor,
                           metadata: torch.Tensor, saliency_map: np.ndarray,
                           device: torch.device = DEVICE) -> Dict:
    """
    Average Drop %: prediction drop after masking top-k% important regions.
    Average Increase %: cases where masking INCREASES prediction (bad sign).
    """
    model.eval()
    image = image.to(device)
    metadata = metadata.to(device)

    # Original prediction
    orig_pred = F.softmax(model(image, metadata), dim=-1)[0, 1].item()

    # Mask top 20% most important pixels
    threshold = np.percentile(saliency_map, 80)
    mask = (saliency_map >= threshold).astype(np.float32)
    mask_tensor = torch.tensor(mask, dtype=torch.float32).to(device)[None, None]

    masked_img = image * (1 - mask_tensor)
    masked_pred = F.softmax(model(masked_img, metadata), dim=-1)[0, 1].item()

    avg_drop = max(0, (orig_pred - masked_pred) / (orig_pred + 1e-8)) * 100
    avg_increase = 1.0 if masked_pred > orig_pred else 0.0

    return {
        'average_drop': avg_drop,
        'average_increase': avg_increase,
        'original_prob': orig_pred,
        'masked_prob': masked_pred,
    }


# ============================================================
# SECTION 21: SHAP for Metadata
# ============================================================

def compute_shap_metadata(model: nn.Module, meta_matrix: np.ndarray,
                           feature_names: List[str],
                           device: torch.device = DEVICE,
                           n_background: int = 50,
                           n_explain: int = 100) -> np.ndarray:
    """
    SHAP values for metadata features using KernelSHAP.
    Reveals which clinical features drive predictions most.

    IMPORTANT: We use a representative background (not zeros/black images)
    to avoid attributing importance to features that differ from a
    meaningless baseline.
    """
    logger.info("Computing SHAP values for metadata features...")

    def predict_fn(meta_np: np.ndarray) -> np.ndarray:
        model.eval()
        # Use mean image as fixed image baseline for metadata SHAP
        dummy_img = torch.zeros(len(meta_np), 3, CFG['img_size'], CFG['img_size']).to(device)
        meta_t = torch.tensor(meta_np, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(dummy_img, meta_t)
            probs = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
        return probs

    background = meta_matrix[:min(n_background, len(meta_matrix))]
    explain_data = meta_matrix[:min(n_explain, len(meta_matrix))]

    explainer = shap.KernelExplainer(predict_fn, background)
    shap_values = explainer.shap_values(explain_data, nsamples=50)

    return shap_values


# ============================================================
# SECTION 22: Comprehensive Visualization
# ============================================================

def plot_training_curves(history: Dict, save_path: str):
    """Publication-quality training curves."""
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle('DaViT + Gated Fusion: Training History', fontsize=14, fontweight='bold')

    epochs = range(1, len(history['train_loss']) + 1)

    axes[0].plot(epochs, history['train_loss'], label='Train', color='#2196F3', linewidth=2)
    axes[0].plot(epochs, history['val_loss'], label='Validation', color='#F44336', linewidth=2)
    axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['train_auc'], label='Train AUC', color='#4CAF50', linewidth=2)
    axes[1].plot(epochs, history['val_auc'], label='Val AUC', color='#FF9800', linewidth=2)
    axes[1].set_title('ROC-AUC'); axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC')
    axes[1].set_ylim([0.5, 1.0]); axes[1].legend(); axes[1].grid(alpha=0.3)

    axes[2].plot(epochs, history['lr'], color='#9C27B0', linewidth=2)
    axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('LR'); axes[2].set_yscale('log'); axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    logger.info(f"Training curves saved: {save_path}")


def plot_roc_pr_curves(probs: np.ndarray, labels: np.ndarray, save_path: str):
    """ROC and PR curves."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('DaViT + Gated Fusion: Performance Curves', fontsize=14, fontweight='bold')

    # ROC
    fpr, tpr, _ = roc_curve(labels, probs)
    auc = roc_auc_score(labels, probs)
    axes[0].plot(fpr, tpr, color='#2196F3', linewidth=2, label=f'ROC AUC = {auc:.4f}')
    axes[0].plot([0,1], [0,1], 'k--', alpha=0.5)
    axes[0].fill_between(fpr, tpr, alpha=0.1, color='#2196F3')
    axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

    # PR
    prec, rec, _ = precision_recall_curve(labels, probs)
    pr_auc = average_precision_score(labels, probs)
    baseline = labels.mean()
    axes[1].plot(rec, prec, color='#4CAF50', linewidth=2, label=f'PR AUC = {pr_auc:.4f}')
    axes[1].axhline(y=baseline, color='#F44336', linestyle='--', label=f'Baseline = {baseline:.3f}')
    axes[1].fill_between(rec, prec, alpha=0.1, color='#4CAF50')
    axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall Curve'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def plot_confusion_matrix(cm: np.ndarray, save_path: str):
    """Annotated confusion matrix."""
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Non-Melanoma', 'Melanoma'],
                yticklabels=['Non-Melanoma', 'Melanoma'],
                ax=ax, cbar=True, linewidths=0.5)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('Actual', fontsize=12)
    ax.set_title('DaViT + Gated Fusion: Confusion Matrix', fontsize=13, fontweight='bold')

    # Add text annotations
    total = cm.sum()
    for i in range(2):
        for j in range(2):
            ax.text(j+0.5, i+0.7, f'{cm[i,j]/total*100:.1f}%',
                    ha='center', va='center', fontsize=9, color='gray')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def plot_calibration(probs: np.ndarray, labels: np.ndarray,
                     calibrated_probs: Optional[np.ndarray] = None,
                     save_path: str = None):
    """Reliability diagram + ECE."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Calibration Analysis', fontsize=13, fontweight='bold')

    # Reliability diagram
    frac_pos, mean_pred = calibration_curve(labels, probs, n_bins=10)
    ece = compute_ece(probs, labels)
    axes[0].plot(mean_pred, frac_pos, 's-', color='#2196F3', label=f'Uncalibrated (ECE={ece:.4f})')

    if calibrated_probs is not None:
        frac_pos_cal, mean_pred_cal = calibration_curve(labels, calibrated_probs, n_bins=10)
        ece_cal = compute_ece(calibrated_probs, labels)
        axes[0].plot(mean_pred_cal, frac_pos_cal, 's-', color='#4CAF50',
                     label=f'Calibrated (ECE={ece_cal:.4f})')

    axes[0].plot([0,1], [0,1], 'k--', label='Perfect calibration')
    axes[0].fill_between([0,1], [0,1], [0,1], alpha=0.1)
    axes[0].set_xlabel('Mean Predicted Probability')
    axes[0].set_ylabel('Fraction of Positives')
    axes[0].set_title('Reliability Diagram'); axes[0].legend(); axes[0].grid(alpha=0.3)

    # Confidence histogram
    axes[1].hist(probs[labels==0], bins=30, alpha=0.7, color='#2196F3', label='Non-Melanoma', density=True)
    axes[1].hist(probs[labels==1], bins=30, alpha=0.7, color='#F44336', label='Melanoma', density=True)
    axes[1].axvline(x=0.5, color='k', linestyle='--', alpha=0.5)
    axes[1].set_xlabel('Predicted Probability')
    axes[1].set_ylabel('Density')
    axes[1].set_title('Confidence Distribution'); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def plot_gradcam_grid(model: nn.Module, dataset: ISICMultimodalDataset,
                      device: torch.device, save_path: str, n_samples: int = 8):
    """
    Grad-CAM overlay grid for publication figures.

    IMPORTANT CAVEAT (for paper):
    Attention/gradient-based maps provide INTERPRETABILITY SIGNALS,
    not proof of causal importance. These visualizations indicate
    WHICH regions influenced the model's decision, but do not
    guarantee alignment with clinical reasoning. Always validate
    with quantitative XAI metrics (insertion/deletion).
    """
    # Get target layer
    target_layer = model.image_encoder.norm
    gradcam = GradCAM(model, target_layer)

    indices = random.sample(range(len(dataset)), min(n_samples, len(dataset)))
    n_cols = 4
    n_rows = (n_samples + n_cols - 1) // n_cols * 2  # 2 rows per sample (original + CAM)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*3, n_rows*3))
    fig.suptitle('Grad-CAM: Interpretability Signals\n(Note: Grad-CAM indicates model attention, '
                 'not clinical proof)', fontsize=11, fontweight='bold')

    IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
    IMAGENET_STD = np.array([0.229, 0.224, 0.225])

    for plot_idx, sample_idx in enumerate(indices):
        row = (plot_idx // n_cols) * 2
        col = plot_idx % n_cols

        sample = dataset[sample_idx]
        image = sample['image'].unsqueeze(0).to(device)
        metadata = sample['metadata'].unsqueeze(0).to(device)
        label = sample['label'].item()

        # Compute CAM
        try:
            cam = gradcam(image, metadata, target_class=1)
        except Exception:
            cam = np.zeros((CFG['img_size'], CFG['img_size']))

        # Original image (denormalized)
        img_np = sample['image'].permute(1, 2, 0).cpu().numpy()
        img_np = np.clip(img_np * IMAGENET_STD + IMAGENET_MEAN, 0, 1)

        # Prediction
        with torch.no_grad():
            logits = model(image, metadata)
            prob = F.softmax(logits, dim=-1)[0, 1].item()

        title_color = '#F44336' if label == 1 else '#2196F3'
        label_name = 'MEL' if label == 1 else 'NV'

        axes[row][col].imshow(img_np)
        axes[row][col].set_title(f'{label_name} | P={prob:.2f}',
                                  fontsize=9, color=title_color)
        axes[row][col].axis('off')

        # CAM overlay
        heatmap = cv2.applyColorMap((cam * 255).astype(np.uint8), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB) / 255.0
        overlay = 0.5 * img_np + 0.5 * heatmap

        axes[row+1][col].imshow(np.clip(overlay, 0, 1))
        axes[row+1][col].set_title('Grad-CAM', fontsize=8, color='gray')
        axes[row+1][col].axis('off')

    # Hide unused axes
    for r in range(n_rows):
        for c in range(n_cols):
            if r // 2 * n_cols + c >= n_samples:
                axes[r][c].axis('off')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    gradcam.remove_hooks()
    logger.info(f"Grad-CAM grid saved: {save_path}")


def plot_uncertainty_analysis(uncertainty_results: Dict, save_path: str):
    """Visualize MC Dropout uncertainty distributions."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle('MC Dropout Uncertainty Estimation', fontsize=13, fontweight='bold')

    mean_probs = uncertainty_results['mean_probs']
    std_probs = uncertainty_results['std_probs']
    entropy = uncertainty_results['entropy']
    labels = uncertainty_results.get('labels', np.zeros(len(mean_probs)))

    # Uncertainty vs Confidence
    axes[0].scatter(mean_probs[labels==0], std_probs[labels==0],
                    alpha=0.5, c='#2196F3', s=20, label='Non-Melanoma')
    axes[0].scatter(mean_probs[labels==1], std_probs[labels==1],
                    alpha=0.5, c='#F44336', s=20, label='Melanoma')
    axes[0].set_xlabel('Mean Probability')
    axes[0].set_ylabel('Std (Uncertainty)')
    axes[0].set_title('Uncertainty vs Confidence')
    axes[0].legend(); axes[0].grid(alpha=0.3)

    # Entropy distribution
    axes[1].hist(entropy[labels==0], bins=30, alpha=0.7, color='#2196F3',
                 label='Non-Melanoma', density=True)
    axes[1].hist(entropy[labels==1], bins=30, alpha=0.7, color='#F44336',
                 label='Melanoma', density=True)
    axes[1].set_xlabel('Predictive Entropy')
    axes[1].set_ylabel('Density')
    axes[1].set_title('Uncertainty Distribution')
    axes[1].legend(); axes[1].grid(alpha=0.3)

    # Calibration error at different uncertainty levels
    uncertainties = np.linspace(0, std_probs.max(), 10)
    coverages, errors = [], []
    for thresh in uncertainties:
        mask = std_probs <= thresh
        if mask.sum() < 5:
            continue
        coverages.append(mask.mean())
        preds = (mean_probs[mask] >= 0.5).astype(int)
        acc = accuracy_score(labels[mask].astype(int), preds)
        errors.append(1 - acc)

    if coverages:
        axes[2].plot(coverages, errors, 'o-', color='#9C27B0', linewidth=2)
        axes[2].set_xlabel('Coverage (fraction of samples)')
        axes[2].set_ylabel('Error Rate')
        axes[2].set_title('Accuracy-Coverage Tradeoff')
        axes[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def plot_shap_beeswarm(shap_values: np.ndarray, feature_names: List[str],
                       save_path: str):
    """SHAP beeswarm plot for metadata feature importance."""
    fig, ax = plt.subplots(figsize=(10, 6))

    # Sort by mean absolute SHAP
    mean_abs = np.abs(shap_values).mean(axis=0)
    sorted_idx = np.argsort(mean_abs)[::-1]
    top_k = min(len(feature_names), 15)
    top_idx = sorted_idx[:top_k]

    for i, feat_idx in enumerate(reversed(top_idx)):
        vals = shap_values[:, feat_idx]
        # Jitter for beeswarm effect
        y_jitter = np.random.uniform(-0.3, 0.3, len(vals))
        scatter = ax.scatter(vals, np.full(len(vals), i) + y_jitter,
                             c=vals, cmap='RdBu_r', alpha=0.6, s=20,
                             vmin=-np.abs(vals).max(), vmax=np.abs(vals).max())

    ax.set_yticks(range(top_k))
    ax.set_yticklabels([feature_names[i] for i in reversed(top_idx)],
                        fontsize=9)
    ax.axvline(x=0, color='k', linestyle='-', alpha=0.3)
    ax.set_xlabel('SHAP Value (impact on model output)', fontsize=11)
    ax.set_title('SHAP Feature Importance: Clinical Metadata', fontsize=12, fontweight='bold')
    ax.grid(alpha=0.3)
    plt.colorbar(scatter, ax=ax, label='Feature Value (normalized)')

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


def plot_gate_analysis(gate_values_list: List[Dict], labels: np.ndarray, save_path: str):
    """
    Gated Fusion gate activation analysis — Multimodal XAI.
    Reveals how much each modality was trusted per sample.
    """
    img_contrib = np.array([g['img_contribution'].mean().item()
                             for g in gate_values_list])
    meta_contrib = np.array([g['meta_contribution'].mean().item()
                              for g in gate_values_list])

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Gated Fusion: Modality Contribution Analysis (Multimodal XAI)',
                 fontsize=12, fontweight='bold')

    # Scatter: image vs metadata contribution
    scatter = axes[0].scatter(img_contrib, meta_contrib,
                               c=labels, cmap='RdBu', alpha=0.6, s=30)
    axes[0].set_xlabel('Image Encoder Contribution')
    axes[0].set_ylabel('Metadata Encoder Contribution')
    axes[0].set_title('Modality Contributions per Sample')
    plt.colorbar(scatter, ax=axes[0], label='Label (0=NV, 1=MEL)')
    axes[0].grid(alpha=0.3)

    # Box plots by class
    nv_mask = labels == 0
    mel_mask = labels == 1
    data = [img_contrib[nv_mask], img_contrib[mel_mask],
            meta_contrib[nv_mask], meta_contrib[mel_mask]]
    bp = axes[1].boxplot(data, patch_artist=True, notch=True,
                          labels=['Img NV', 'Img MEL', 'Meta NV', 'Meta MEL'])
    colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    axes[1].set_ylabel('Contribution Magnitude')
    axes[1].set_title('Modality Contribution by Diagnosis')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


# ============================================================
# SECTION 23: Ablation Study
# ============================================================

class AblationStudy:
    """
    Structured ablation study for publication-quality analysis.
    Tests the contribution of each pipeline component.
    """

    def __init__(self, model_class, cfg: Dict, base_model_state: Dict,
                 val_loader: DataLoader, device: torch.device):
        self.model_class = model_class
        self.cfg = cfg
        self.base_state = base_model_state
        self.val_loader = val_loader
        self.device = device
        self.results = {}

    def run_variant(self, name: str, model: nn.Module) -> Dict:
        """Evaluate a model variant."""
        trainer = Trainer(model, self.cfg, self.device)
        metrics = trainer.evaluate(self.val_loader, use_ema=False)
        self.results[name] = {
            'auc': metrics['auc'],
            'sensitivity': metrics.get('sensitivity', 0),
            'specificity': metrics.get('specificity', 0),
            'f1': metrics.get('f1', 0),
        }
        logger.info(f"Ablation [{name}]: AUC={metrics['auc']:.4f}, "
                    f"Sens={metrics.get('sensitivity', 0):.4f}")
        return self.results[name]

    def plot_results(self, save_path: str):
        """Bar chart of ablation results."""
        if not self.results:
            return

        names = list(self.results.keys())
        aucs = [self.results[n]['auc'] for n in names]
        sens = [self.results[n]['sensitivity'] for n in names]

        fig, axes = plt.subplots(1, 2, figsize=(14, 6))
        fig.suptitle('Ablation Study: Component Importance', fontsize=13, fontweight='bold')

        colors = ['#2196F3' if i == 0 else '#90CAF9' for i in range(len(names))]
        axes[0].barh(names, aucs, color=colors)
        axes[0].set_xlabel('ROC-AUC')
        axes[0].set_title('AUC Comparison')
        axes[0].axvline(x=aucs[0], color='red', linestyle='--', alpha=0.7)
        axes[0].grid(axis='x', alpha=0.3)
        for i, v in enumerate(aucs):
            axes[0].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

        axes[1].barh(names, sens, color=['#4CAF50' if i == 0 else '#A5D6A7'
                                          for i in range(len(names))])
        axes[1].set_xlabel('Sensitivity (Melanoma Recall)')
        axes[1].set_title('Melanoma Sensitivity Comparison')
        axes[1].axvline(x=sens[0], color='red', linestyle='--', alpha=0.7)
        axes[1].grid(axis='x', alpha=0.3)
        for i, v in enumerate(sens):
            axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)

        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        plt.close()


# ============================================================
# SECTION 24: Error Analysis
# ============================================================

def error_analysis(probs: np.ndarray, labels: np.ndarray,
                   images: List[str], dataset: ISICMultimodalDataset,
                   save_path: str):
    """
    Analyze failure cases:
    - False Positives (FP): Non-melanoma predicted as melanoma
    - False Negatives (FN): Melanoma missed — CLINICALLY MOST DANGEROUS
    - High uncertainty predictions

    CLINICAL IMPORTANCE of FN reduction:
    A missed melanoma (FN) is significantly more harmful than a false alarm (FP).
    FP → unnecessary biopsy (patient anxiety, minor procedure)
    FN → delayed diagnosis → worse prognosis, potential death
    Any deployed melanoma screening AI must prioritize minimizing FN.
    """
    preds = (probs >= 0.5).astype(int)

    fp_idx = np.where((preds == 1) & (labels == 0))[0]
    fn_idx = np.where((preds == 0) & (labels == 1))[0]
    tp_idx = np.where((preds == 1) & (labels == 1))[0]
    tn_idx = np.where((preds == 0) & (labels == 0))[0]

    logger.info("\n" + "=" * 50)
    logger.info("ERROR ANALYSIS")
    logger.info("=" * 50)
    logger.info(f"True Positives:  {len(tp_idx)} (correct melanoma detection)")
    logger.info(f"True Negatives:  {len(tn_idx)} (correct non-melanoma)")
    logger.info(f"False Positives: {len(fp_idx)} (unnecessary alarm — patient anxiety risk)")
    logger.info(f"False Negatives: {len(fn_idx)} (MISSED MELANOMA — critical clinical risk)")
    logger.info(f"FN Rate: {len(fn_idx)/(len(fn_idx)+len(tp_idx)+1e-8):.4f}")

    # High confidence errors
    high_conf_fp = fp_idx[probs[fp_idx] > 0.8] if len(fp_idx) > 0 else []
    high_conf_fn = fn_idx[probs[fn_idx] < 0.2] if len(fn_idx) > 0 else []

    logger.info(f"\nHigh-confidence FP (>0.8): {len(high_conf_fp)} — model confidently misclassifies benign")
    logger.info(f"High-confidence FN (<0.2): {len(high_conf_fn)} — model misses melanoma with high confidence")

    if len(high_conf_fn) > 0:
        logger.warning("CRITICAL: High-confidence false negatives detected. "
                       "These cases require manual review and potential dataset bias analysis.")

    # Error probability distribution
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    fig.suptitle('Error Analysis: Failure Case Distribution', fontsize=12, fontweight='bold')

    if len(fp_idx) > 0:
        axes[0].hist(probs[fp_idx], bins=20, color='#FF9800', alpha=0.8, density=True)
    axes[0].set_xlabel('Predicted Probability')
    axes[0].set_title(f'False Positive Distribution (n={len(fp_idx)})')
    axes[0].axvline(x=0.5, color='r', linestyle='--')
    axes[0].grid(alpha=0.3)

    if len(fn_idx) > 0:
        axes[1].hist(probs[fn_idx], bins=20, color='#F44336', alpha=0.8, density=True)
    axes[1].set_xlabel('Predicted Probability')
    axes[1].set_title(f'False Negative Distribution (n={len(fn_idx)}) — CLINICALLY CRITICAL')
    axes[1].axvline(x=0.5, color='r', linestyle='--')
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()


# ============================================================
# SECTION 25: FULL PIPELINE EXECUTION
# ============================================================

def run_full_pipeline():
    """
    Execute the complete DaViT + Gated Fusion pipeline.
    All steps are logged, all outputs are saved to CFG['output_dir'].
    """
    logger.info("=" * 70)
    logger.info("STARTING: Multimodal Skin Cancer AI Pipeline")
    logger.info("Model: DaViT-Tiny + Gated Fusion")
    logger.info("=" * 70)

    # ── Step 1: Load Data ────────────────────────────────────
    logger.info("\n[1/12] Loading dataset from ZIP...")
    loader = ZipDataLoader(CFG['data_root'])
    splits = loader.load_all()

    # ── Step 2: Fit Metadata Preprocessor ───────────────────
    logger.info("\n[2/12] Fitting metadata preprocessor on training data only...")
    meta_preprocessor.fit(splits['train']['df'])
    meta_dim = meta_preprocessor.output_dim

    # ── Step 3: Build Datasets & DataLoaders ────────────────
    logger.info("\n[3/12] Building datasets and dataloaders...")
    train_transform = MedicalAugmentations.get_train_transform(CFG['img_size'])
    val_transform = MedicalAugmentations.get_val_transform(CFG['img_size'])

    train_dataset = ISICMultimodalDataset(
        splits['train'], meta_preprocessor, transform=train_transform,
        img_size=CFG['img_size'], apply_medical_preprocessing=True)
    val_dataset = ISICMultimodalDataset(
        splits['val'], meta_preprocessor, transform=val_transform,
        img_size=CFG['img_size'], apply_medical_preprocessing=True)
    test_dataset = ISICMultimodalDataset(
        splits['test'], meta_preprocessor, transform=val_transform,
        img_size=CFG['img_size'], apply_medical_preprocessing=True)

    # Class weighting for imbalance
    matched_df = train_dataset.df.iloc[train_dataset.valid_indices]
    train_labels = matched_df['class'].values 
    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / (class_counts + 1)
    sample_weights = class_weights[train_labels]
    sampler = WeightedRandomSampler(
        weights=sample_weights, num_samples=len(sample_weights), replacement=True)

    train_loader = DataLoader(train_dataset, batch_size=CFG['batch_size'],
                              sampler=sampler, num_workers=0, pin_memory=True,
                              drop_last=True)
    val_loader = DataLoader(val_dataset, batch_size=CFG['batch_size'],
                            shuffle=False, num_workers=0, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=CFG['batch_size'],
                             shuffle=False, num_workers=0, pin_memory=True)

    logger.info(f"Train: {len(train_dataset)} | Val: {len(val_dataset)} | "
                f"Test: {len(test_dataset)}")
    logger.info(f"Class distribution — Melanoma: {class_counts[1] if len(class_counts)>1 else 0}, "
                f"Non-Melanoma: {class_counts[0]}")

    # ── Step 4: Build Model ──────────────────────────────────
    logger.info("\n[4/12] Building DaViT-Tiny + Gated Fusion model...")
    model = DaViTGatedFusionModel(metadata_input_dim=meta_dim, cfg=CFG)
    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    logger.info(f"Model parameters: {n_params:.2f}M")

    # ── Step 5: Training ─────────────────────────────────────
    logger.info("\n[5/12] Training model...")
    trainer = Trainer(model, CFG, DEVICE)
    history = trainer.fit(train_loader, val_loader)

    # Load best checkpoint
    ckpt_path = f"{CFG['output_dir']}/checkpoints/best_model.pth"
    if Path(ckpt_path).exists():
        ckpt = torch.load(ckpt_path, map_location=DEVICE)
        model.load_state_dict(ckpt['model_state'])
        trainer.ema.shadow.load_state_dict(ckpt['ema_state'])
        logger.info(f"Loaded best checkpoint (Val AUC: {ckpt['val_auc']:.4f})")

    # ── Step 6: Evaluation ───────────────────────────────────
    logger.info("\n[6/12] Evaluating on test set...")
    test_metrics = trainer.evaluate(test_loader, use_ema=True)
    full_metrics = compute_full_metrics(test_metrics['probs'], test_metrics['labels'])
    print_metrics_table(full_metrics, "TEST SET EVALUATION — DaViT + Gated Fusion")

    # ── Step 7: Calibration ──────────────────────────────────
    logger.info("\n[7/12] Temperature scaling calibration...")
    val_metrics = trainer.evaluate(val_loader, use_ema=True)

    # Collect val logits for calibration
    model.eval()
    val_logits_list = []
    val_labels_list = []
    with torch.no_grad():
        for batch in val_loader:
            imgs = batch['image'].to(DEVICE)
            meta = batch['metadata'].to(DEVICE)
            lbls = batch['label']
            logits = trainer.ema.shadow(imgs, meta)
            val_logits_list.append(logits.cpu())
            val_labels_list.append(lbls)

    val_logits_all = torch.cat(val_logits_list)
    val_labels_all = torch.cat(val_labels_list)

    temp_scaler = TemperatureScaling()
    temp = temp_scaler.fit(val_logits_all, val_labels_all)

    # Calibrated test predictions
    test_logits_list = []
    with torch.no_grad():
        for batch in test_loader:
            imgs = batch['image'].to(DEVICE)
            meta = batch['metadata'].to(DEVICE)
            logits = trainer.ema.shadow(imgs, meta)
            test_logits_list.append(logits.cpu())
    test_logits_all = torch.cat(test_logits_list)
    calibrated_probs = F.softmax(temp_scaler(test_logits_all), dim=-1)[:, 1].numpy()

    ece_before = compute_ece(test_metrics['probs'], test_metrics['labels'])
    ece_after = compute_ece(calibrated_probs, test_metrics['labels'])
    logger.info(f"ECE Before: {ece_before:.4f} → After: {ece_after:.4f}")

    # ── Step 8: MC Dropout Uncertainty ───────────────────────
    logger.info("\n[8/12] MC Dropout uncertainty estimation...")
    all_uncertainty = {'mean_probs': [], 'std_probs': [], 'entropy': []}
    unc_labels = []

    for batch in test_loader:
        unc_result = mc_dropout_predict(model, batch, n_samples=CFG['mc_dropout_samples'])
        all_uncertainty['mean_probs'].extend(unc_result['mean_probs'].tolist())
        all_uncertainty['std_probs'].extend(unc_result['std_probs'].tolist())
        all_uncertainty['entropy'].extend(unc_result['entropy'].tolist())
        unc_labels.extend(batch['label'].numpy().tolist())

    for k in all_uncertainty:
        all_uncertainty[k] = np.array(all_uncertainty[k])
    all_uncertainty['labels'] = np.array(unc_labels)

    # ── Step 9: TTA ──────────────────────────────────────────
    logger.info("\n[9/12] Test-Time Augmentation...")
    tta_transforms = MedicalAugmentations.get_tta_transforms(CFG['img_size'], n=CFG['tta_steps'])
    tta_probs = tta_predict(trainer.ema.shadow, test_dataset, tta_transforms, DEVICE)
    tta_auc = roc_auc_score(test_metrics['labels'], tta_probs) if len(set(test_metrics['labels'])) > 1 else 0.5
    logger.info(f"TTA AUC: {tta_auc:.4f} (vs base: {full_metrics['roc_auc']:.4f})")

    # ── Step 10: XAI ─────────────────────────────────────────
    logger.info("\n[10/12] Computing XAI explanations...")

    # Grad-CAM grid
    plot_gradcam_grid(
        model=trainer.ema.shadow,
        dataset=test_dataset,
        device=DEVICE,
        save_path=f"{CFG['output_dir']}/figures/gradcam_grid.png",
        n_samples=8
    )

    # Quantitative XAI on a subset
    logger.info("Computing quantitative XAI metrics (insertion/deletion)...")
    target_layer = trainer.ema.shadow.image_encoder.norm
    gradcam = GradCAM(trainer.ema.shadow, target_layer)
    gradcam_pp = GradCAMPlusPlus(trainer.ema.shadow, target_layer)

    qxai_results = {'gradcam': [], 'gradcam_pp': [], 'ig': []}
    xai_sample_indices = random.sample(range(len(test_dataset)),
                                        min(CFG['xai_samples'], len(test_dataset)))

    for idx in xai_sample_indices[:5]:  # Fast demo: 5 samples
        sample = test_dataset[idx]
        image = sample['image'].unsqueeze(0).to(DEVICE)
        metadata = sample['metadata'].unsqueeze(0).to(DEVICE)

        try:
            cam = gradcam(image.clone(), metadata.clone())
            ins_del = insertion_deletion_metrics(trainer.ema.shadow, image, metadata, cam, n_steps=20)
            avg_drop_r = average_drop_increase(trainer.ema.shadow, image, metadata, cam)
            qxai_results['gradcam'].append({**ins_del, **avg_drop_r})
        except Exception as e:
            logger.warning(f"XAI failed for sample {idx}: {e}")

    if qxai_results['gradcam']:
        avg_ins = np.mean([r['insertion_auc'] for r in qxai_results['gradcam']])
        avg_del = np.mean([r['deletion_auc'] for r in qxai_results['gradcam']])
        avg_drop_val = np.mean([r['average_drop'] for r in qxai_results['gradcam']])
        logger.info(f"Quantitative XAI — Grad-CAM:")
        logger.info(f"  Insertion AUC: {avg_ins:.4f}")
        logger.info(f"  Deletion AUC:  {avg_del:.4f}")
        logger.info(f"  Average Drop:  {avg_drop_val:.2f}%")
        logger.info(f"  Faithfulness Gap: {avg_ins - avg_del:.4f}")

    gradcam.remove_hooks()
    gradcam_pp.remove_hooks()

    # SHAP for metadata
    logger.info("Computing SHAP values for metadata...")
    feature_names = meta_preprocessor.available_num + meta_preprocessor.available_cat
    try:
        shap_vals = compute_shap_metadata(
            trainer.ema.shadow,
            meta_preprocessor.transform(splits['test']['df']),
            feature_names, DEVICE,
            n_background=30, n_explain=50
        )
        plot_shap_beeswarm(shap_vals, feature_names,
                           f"{CFG['output_dir']}/figures/shap_metadata.png")
    except Exception as e:
        logger.warning(f"SHAP computation failed: {e}")

    # Gate analysis
    gate_values_list = []
    gate_labels = []
    trainer.ema.shadow.eval()
    with torch.no_grad():
        for batch in test_loader:
            imgs = batch['image'].to(DEVICE)
            meta = batch['metadata'].to(DEVICE)
            _, gates = trainer.ema.shadow(imgs, meta, return_gates=True)
            gate_values_list.extend([{k: v[i] for k, v in gates.items()}
                                     for i in range(len(imgs))])
            gate_labels.extend(batch['label'].numpy().tolist())

    plot_gate_analysis(gate_values_list, np.array(gate_labels),
                       f"{CFG['output_dir']}/figures/gate_analysis.png")

    # ── Step 11: Visualizations ───────────────────────────────
    logger.info("\n[11/12] Generating publication figures...")

    plot_training_curves(history, f"{CFG['output_dir']}/figures/training_curves.png")
    plot_roc_pr_curves(test_metrics['probs'], test_metrics['labels'],
                       f"{CFG['output_dir']}/figures/roc_pr_curves.png")
    plot_confusion_matrix(full_metrics['confusion_matrix'],
                          f"{CFG['output_dir']}/figures/confusion_matrix.png")
    plot_calibration(test_metrics['probs'], test_metrics['labels'],
                     calibrated_probs=calibrated_probs,
                     save_path=f"{CFG['output_dir']}/figures/calibration.png")
    plot_uncertainty_analysis(all_uncertainty,
                              f"{CFG['output_dir']}/figures/uncertainty.png")

    # Error analysis
    error_analysis(
        test_metrics['probs'], test_metrics['labels'],
        [test_dataset.df.iloc[i]['image'] for i in test_dataset.valid_indices],
        test_dataset,
        f"{CFG['output_dir']}/figures/error_analysis.png"
    )

    # ── Step 12: Save Summary ─────────────────────────────────
    logger.info("\n[12/12] Saving results summary...")
    summary = {
        'model': 'DaViT-Tiny + Gated Fusion',
        'parameters_M': round(n_params, 2),
        'temperature': round(temp, 4),
        'test_metrics': {
            k: round(float(v), 4) for k, v in full_metrics.items()
            if isinstance(v, (float, int, np.floating)) and k != 'confusion_matrix'
        },
        'tta_auc': round(float(tta_auc), 4),
        'ece_before': round(float(ece_before), 4),
        'ece_after': round(float(ece_after), 4),
        'best_val_auc': round(float(trainer.best_val_auc), 4),
    }

    with open(f"{CFG['output_dir']}/results_summary.json", 'w') as f:
        json.dump(summary, f, indent=2)

    logger.info("\n" + "=" * 70)
    logger.info("PIPELINE COMPLETE — DaViT + Gated Fusion")
    logger.info(f"Test AUC:       {full_metrics['roc_auc']:.4f}")
    logger.info(f"Test Sensitivity: {full_metrics.get('sensitivity', 0):.4f}")
    logger.info(f"Test Specificity: {full_metrics.get('specificity', 0):.4f}")
    logger.info(f"TTA AUC:        {tta_auc:.4f}")
    logger.info(f"ECE (after cal): {ece_after:.4f}")
    logger.info(f"Output dir: {CFG['output_dir']}")
    logger.info("=" * 70)

    return model, trainer, summary


# ============================================================
# SECTION 26: Discussion & Scientific Notes
# ============================================================
#
# STRENGTHS:
# 1. Dual-attention (channel + spatial) captures multi-scale melanoma features
# 2. Gated fusion adapts modality weighting to individual patient data quality
# 3. SAM optimizer improves OOD generalization to new dermoscopy devices
# 4. Temperature calibration aligns predicted probabilities with clinical reality
# 5. MC dropout provides uncertainty quantification for triage decisions
#
# WEAKNESSES:
# 1. DaViT-Tiny trained from scratch — would benefit from ImageNet pre-training
# 2. Gated fusion assumes independent modalities — cross-modal interactions
#    are only partially captured
# 3. MC Dropout is an approximation of Bayesian inference — uncertainty estimates
#    may be overconfident when training and test distributions differ significantly
#
# ATTENTION DISCLAIMER:
# The Grad-CAM and attention maps visualized in this notebook indicate regions
# that influenced the model's decision through gradient flow. They do NOT
# constitute clinical proof that these regions contain diagnostic features.
# Models can produce high gradient responses for non-diagnostic regions
# (e.g., hair artifacts, border noise) if not properly regularized.
# Quantitative XAI evaluation (insertion/deletion metrics) is MANDATORY
# to validate the clinical relevance of explanations.
#
# DEPLOYMENT LIMITATIONS:
# 1. This model was trained on ISIC data — performance on external dermoscopy
#    devices and clinical settings needs prospective validation
# 2. Calibration may drift over time as patient population shifts
# 3. The model does not replace dermatologist judgment — it is a screening tool
# 4. Fairness evaluation across skin tones, ages, and anatomical sites is
#    required before any clinical deployment

# ============================================================
# MAIN ENTRY POINT
# ============================================================

if __name__ == '__main__':
    model, trainer, summary = run_full_pipeline()
    print("\nFinal Results Summary:")
    print(json.dumps({k: v for k, v in summary.items()
                      if k != 'test_metrics'}, indent=2))
    print("\nTest Metrics:")
    for k, v in summary['test_metrics'].items():
        print(f"  {k}: {v}")

2026-05-12 19:34:00,631 - INFO - Device: cuda | PyTorch: 2.10.0+cu128
2026-05-12 19:34:00,658 - INFO - ======================================================================
2026-05-12 19:34:00,659 - INFO - STARTING: Multimodal Skin Cancer AI Pipeline
2026-05-12 19:34:00,659 - INFO - Model: DaViT-Tiny + Gated Fusion
2026-05-12 19:34:00,660 - INFO - ======================================================================
2026-05-12 19:34:00,661 - INFO - 
[1/12] Loading dataset from ZIP...
2026-05-12 19:34:00,669 - WARNING - ZIP not found: /kaggle/input/datasets/ahmedmohsen2005/final-dataset/train.zip
2026-05-12 19:34:00,671 - INFO - Using pre-extracted directory: /kaggle/input/datasets/ahmedmohsen2005/final-dataset/train
2026-05-12 19:34:19,086 - INFO - CSV loaded: train.csv | Rows: 11941 | Melanoma samples: 5971
2026-05-12 19:34:35,653 - INFO - Valid images found: 11941
2026-05-12 19:34:35,660 - INFO - [train] Matched 0/11941 images
2026-05-12 19:34:35,661 - WARNING - [train] Less than 5

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 